# Respiratory Sound ML: Dataset Preparation, Model Development and Evaluation

This notebook documents the machine-learning development carried out for the dual-microphone respiratory sound project. The work is presented as an experiment sequence rather than a single model run: the data pipeline was audited first, different feature representations and model families were compared, and the final experiments focused on temporal modelling, thresholds, device variation, and failure analysis.

**Task:** classify respiratory cycles into four acoustic categories: normal, crackle, wheeze, and crackle+wheeze.

**Evaluation principle:** patient separation is preserved and class-sensitive metrics, especially macro F1, are used alongside accuracy.

**Results note:** historical experiment aggregates are labelled as recorded evidence when the matching sample-level artifacts are not available for a fresh rerun.

### ML contribution at a glance

- patient-separated dataset construction and split auditing
- log-Mel, delta and delta-delta feature engineering
- fixed-length, time-resized and frame-padded representations
- compact CNN, weighted CNN and patient-balanced CNN comparisons
- CNN embeddings with RBF-SVM, k-NN and bagged k-NN
- CNN–LSTM modelling of crackle and wheeze events
- validation-based threshold selection and four-class mapping
- device-stratified error analysis and training-derived device alignment

## 1. Dataset discovery and annotation parsing

### 1.1 Locate recordings and annotations


In [ ]:
from pathlib import Path

# Faster dataset scan: avoid multiple redundant scans

data_dir = Path("../data/raw")
wav_files = sorted(data_dir.rglob("*.wav"))
txt_files = sorted(data_dir.rglob("*.txt"))
txt_map = {txt.stem: txt for txt in txt_files}

matched_pairs = []
for wav_path in wav_files:
    txt_path = txt_map.get(wav_path.stem)
    if txt_path is not None:
        matched_pairs.append((wav_path, txt_path))

print("WAV files:", len(wav_files))
print("TXT files:", len(txt_files))
print("Matched WAV-TXT pairs:", len(matched_pairs))


### 1.2 Parse respiratory-cycle labels


In [ ]:
# Use vectorized label assignment instead of per-row Python loops
import pandas as pd
import numpy as np

records = []

for wav_path, txt_path in matched_pairs:
    annotation = pd.read_csv(
        txt_path,
        sep="\t",
        header=None,
        names=["start", "end", "crackle", "wheeze"]
    )

    annotation["wav_path"] = str(wav_path)
    annotation["patient"] = wav_path.stem.split("_")[0]
    annotation["recording"] = wav_path.stem

    annotation["label"] = np.select(
        [
            (annotation["crackle"] == 0) & (annotation["wheeze"] == 0),
            (annotation["crackle"] == 0) & (annotation["wheeze"] == 1),
            (annotation["crackle"] == 1) & (annotation["wheeze"] == 0),
        ],
        ["normal", "wheeze", "crackle"],
        default="crackle+wheeze"
    )

    records.append(
        annotation[
            ["wav_path", "patient", "recording", "start", "end", "crackle", "wheeze", "label"]
        ]
    )

annotation_df = pd.concat(records, ignore_index=True)
print("Total annotated segments:", len(annotation_df))
print("\nClass distribution:")
print(annotation_df["label"].value_counts())


### 1.3 Build the cycle-level annotation table


In [ ]:
records = []
for wav_path, txt_path in matched_pairs:
    annotation =pd.read_csv(txt_path, sep="\t",header=None)
    annotation.columns =["start","end","crackle","wheeze"]

    for _, row in annotation.iterrows():
        if row["crackle"] == 0 and row["wheeze"] == 0:
            label = "normal"
        elif row["crackle"] == 0 and row["wheeze"] == 1:
            label = "wheeze"
        elif row["crackle"] == 1 and row["wheeze"] == 0:
             label = "crackle"
        else:
         label = "crackle+wheeze"
records.append({
"wav_path": str(wav_path),"patient": wav_path.name.split("_")[0],
"recordings": wav_path.stem,"start": row["start"],"end": row["end"],"crackle": int(row["crackle"]),"wheeze": int(row["wheeze"]),"label": label})
annotation_df = pd.DataFrame(records)
print("total annotated segments:", len(annotation_df))


### 1.4 Audit patients and recordings


In [ ]:
print("Number of unique patients:")
print(annotation_df["patient"].nunique())


In [ ]:
patient_counts = annotation_df["patient"].value_counts()

print(patient_counts.head(10))


In [ ]:
from pathlib import Path
import pandas as pd
split_path = Path("../data/sp/ICBHI_challenge_train_test.txt")
split_df =pd.read_csv(split_path, sep=r"\s+", header=None,names=["recording","official_split"])
split_df["recording"] = split_df["recording"].map(
    lambda name: Path(str(name)).stem
)
annotation_df["recording"] = annotation_df["recording"].map(
    lambda name: Path(str(name)).stem
)
print("rows in official split file:", len(split_df))
print("unique recordings in official split file:", split_df["recording"].nunique())
print("\nOfficial split file head:")
print(split_df["official_split"].value_counts())
print("\nfirst few recordings:")
print(split_df.head())


In [ ]:
data_df = annotation_df.merge(
    split_df,
    on="recording",
    how="left",
    validate="many_to_one",
    indicator=True
)
print("merge results:")
print(data_df["_merge"].value_counts())
print("\nfirst five rows of merged dataset:")
display(data_df.head())


In [ ]:
unmatched_rows = data_df[data_df["_merge"] == "left_only"]
print("unmatched annotation rows",len(unmatched_rows))
unmatched_recordings = unmatched_rows[["recording", "patient", "wav_path"]].drop_duplicates()
print("unmatched recordings:", len(unmatched_recordings))
display(unmatched_recordings.sort_values("recording"))


In [ ]:
print(unmatched_recordings.to_string(index=False))


In [ ]:
missing_id = unmatched_recordings.iloc[0]["recording"]

print("Unmatched recording:", missing_id)

print(
    "Present in official split:",
    missing_id in set(split_df["recording"])
)


In [ ]:
missing_id = unmatched_recordings.iloc[0]["recording"]
patient_id = missing_id.split("_")[0]

patient_records = split_df[
    split_df["recording"].str.startswith(patient_id + "_")
]

print("Patient ID:", patient_id)
print("Other recordings found:", len(patient_records))

display(patient_records)


In [ ]:
# Get every annotated segment belonging to patient 226
patient226_val = annotation_df[
    annotation_df["patient"].astype(str) == "226"
].copy()

# Mark these recordings as reserved for validation
patient226_val["official_split"] = "reserved_validation"
patient226_val["_merge"] = "reserved_validation"

print("Patient 226 validation segments:", len(patient226_val))
print(
    "Patient 226 validation recordings:",
    patient226_val["recording"].nunique()
)

display(
    patient226_val[
        ["recording", "patient", "label"]
    ].drop_duplicates().sort_values("recording")
)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
# Preserve the official test recordings
test_df = data_df[
    data_df["official_split"] == "test"
].copy()

test_patients = set(test_df["patient"].astype(str))
# Start with official training recordings
train_pool = data_df[
    data_df["official_split"] == "train"
].copy()

train_pool["patient"] = train_pool["patient"].astype(str)

# Remove patients who appear in the official test set
train_pool = train_pool[
    ~train_pool["patient"].isin(test_patients)
].copy()

# Patient 226 is reserved entirely for validation
train_pool = train_pool[
    train_pool["patient"] != "226"
].copy()
# Split the remaining training pool by patient
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_pool,
        groups=train_pool["patient"]
    )
)

train_df = train_pool.iloc[train_idx].copy()
val_other_patients = train_pool.iloc[val_idx].copy()
# Add all patient 226 recordings to validation
val_df = pd.concat(
    [val_other_patients, patient226_val],
    ignore_index=True,
    sort=False
)

# Clean up row numbering
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training segments:", len(train_df))
print("Validation segments:", len(val_df))
print("Test segments:", len(test_df))


In [ ]:
train_patients = set(train_df["patient"].astype(str))
val_patients = set(val_df["patient"].astype(str))
test_patients = set(test_df["patient"].astype(str))

print("Training-validation overlap:",
      train_patients & val_patients)

print("Training-test overlap:",
      train_patients & test_patients)

print("Validation-test overlap:",
      val_patients & test_patients)


## 2. Patient-separated partitioning

The supplied split assignment is reconciled with the annotation table, unmatched recordings are audited, and the development partitions are constructed so patient identities do not cross train, validation, and evaluated test groups.


### 2.1 Audio analysis setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import tensorflow as tf
print(("Numpy:",np.__version__))
print(("Pandas:",pd.__version__))
print(("Librosa:",librosa.__version__))
print(("TensorFlow:",tf.__version__))
print("\nCurrent dataset sizes:")
print("train:", (len(train_df)))
print("val:", (len(val_df)))
print("test:", (len(test_df)))


### 2.2 Inspect cycle-duration distribution


In [ ]:
all_durations = pd.concat(
    [train_df["end"] - train_df["start"],
     val_df["end"] - val_df["start"],
     test_df["end"] - test_df["start"]
], ignore_index=True)
print(all_durations.describe(
    percentiles=[0.50,0.90,0.95,0.99]
))
print("\nsegments longer than 5 seconds:")
print((all_durations > 5).sum())


## 3. Audio standardization and feature extraction

### 3.1 Define the audio and feature contract


In [ ]:
TARGET_SR = 4000

CLIP_SECONDS = 5.0
TARGET_SAMPLES = int(TARGET_SR * CLIP_SECONDS)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64

FMIN = 20
FMAX = 2000

CLASSES = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

class_to_idx = {
    label: idx
    for idx, label in enumerate(CLASSES)
}

idx_to_class = {
    idx: label
    for label, idx in class_to_idx.items()
}

print("Target samples per segment:", TARGET_SAMPLES)
print("Class mapping:", class_to_idx)


### 3.2 Standardize segment length


In [ ]:
def fit_audio_length(y, target_samples):
    """
    Make an audio segment exactly target_samples long.
    Short segments are zero-padded.
    Long segments are center-cropped.
    """

    if len(y) > target_samples:
        start = (len(y) - target_samples) // 2
        y = y[start:start + target_samples]

    elif len(y) < target_samples:
        total_padding = target_samples - len(y)

        left_padding = total_padding // 2
        right_padding = total_padding - left_padding

        y = np.pad(
            y,
            (left_padding, right_padding),
            mode="constant"
        )

    return y.astype(np.float32)


def add_white_noise_at_snr(signal, snr_db, rng):
    """
    Add synthetic white Gaussian noise at a specified SNR.
    SNR is calculated from the signal and noise powers.
    """

    signal_power = np.mean(signal ** 2)

    if signal_power < 1e-12:
        return signal.copy()

    noise = rng.standard_normal(len(signal)).astype(np.float32)

    noise_power = np.mean(noise ** 2)

    desired_noise_power = (
        signal_power / (10 ** (snr_db / 10))
    )

    noise = noise * np.sqrt(
        desired_noise_power / (noise_power + 1e-12)
    )

    return (signal + noise).astype(np.float32)


def extract_logmel(segment):
    """
    Convert a fixed-length audio segment into a normalized
    Log-Mel spectrogram.
    """

    mel_power = librosa.feature.melspectrogram(
        y=segment,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    log_mel = librosa.power_to_db(
        mel_power,
        ref=np.max
    )

    # Keep values within a consistent dynamic range.
    log_mel = np.clip(log_mel, -80.0, 0.0)

    # Map the values from [-80, 0] dB to [0, 1].
    log_mel = (log_mel + 80.0) / 80.0

    return log_mel.astype(np.float32)


### 3.3 Extract log-Mel features


In [ ]:
def build_features(
    frame,
    split_name,
    noise_snr_db=None,
    random_seed=42
):
    """
    Build Log-Mel tensors and integer labels from an annotation table.

    noise_snr_db:
        None            = clean/base audio
        A number        = fixed SNR for each segment
        A list of levels = randomly select one SNR per segment
    """

    rng = np.random.default_rng(random_seed)

    features = []
    labels = []
    metadata = []

    skipped_segments = 0
    cropped_segments = 0

    for recording_number, (wav_path, group) in enumerate(
        frame.groupby("wav_path", sort=False),
        start=1
    ):

        # Load and resample the entire recording to 4 kHz.
        audio, _ = librosa.load(
            str(wav_path),
            sr=TARGET_SR,
            mono=True
        )

        for row in group.itertuples(index=False):

            # Convert annotation times in seconds into sample indices.
            start_sample = int(
                np.floor(float(row.start) * TARGET_SR)
            )

            end_sample = int(
                np.ceil(float(row.end) * TARGET_SR)
            )

            # Keep indices within the available audio range.
            start_sample = max(0, min(start_sample, len(audio)))
            end_sample = max(0, min(end_sample, len(audio)))

            if end_sample <= start_sample:
                skipped_segments += 1
                continue

            # Extract the annotated interval.
            segment = audio[start_sample:end_sample].copy()

            if len(segment) > TARGET_SAMPLES:
                cropped_segments += 1

            # Ensure a fixed input length.
            segment = fit_audio_length(
                segment,
                TARGET_SAMPLES
            )

            actual_snr = np.nan

            # Optional noise injection.
            if noise_snr_db is not None:

                if isinstance(
                    noise_snr_db,
                    (list, tuple, np.ndarray)
                ):
                    actual_snr = float(
                        rng.choice(noise_snr_db)
                    )
                else:
                    actual_snr = float(noise_snr_db)

                segment = add_white_noise_at_snr(
                    segment,
                    actual_snr,
                    rng
                )

            # Extract the Log-Mel representation.
            log_mel = extract_logmel(segment)

            # Add the channel dimension expected by Conv2D.
            features.append(log_mel[..., np.newaxis])

            label = str(row.label).strip().lower()

            if label not in class_to_idx:
                raise ValueError(
                    f"Unknown label {label!r} "
                    f"in recording {row.recording}"
                )

            labels.append(class_to_idx[label])

            metadata.append({
                "wav_path": str(wav_path),
                "recording": str(row.recording),
                "patient": str(row.patient),
                "start": float(row.start),
                "end": float(row.end),
                "label": label,
                "label_id": class_to_idx[label],
                "noise_snr_db": actual_snr
            })

        if recording_number % 100 == 0:
            print(
                f"{split_name}: processed "
                f"{recording_number} recordings"
            )

    if not features:
        raise ValueError(
            f"No valid features produced for {split_name}"
        )

    X = np.stack(features).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    metadata_df = pd.DataFrame(metadata)

    print(f"\n{split_name} complete")
    print("Features shape:", X.shape)
    print("Labels shape:", y.shape)
    print("Invalid segments skipped:", skipped_segments)
    print("Segments center-cropped:", cropped_segments)
    print("Feature memory (MB):", round(X.nbytes / 1e6, 2))

    return X, y, metadata_df


### 3.4 Generate train, validation and test tensors


In [ ]:
X_train, y_train, meta_train = build_features(
    train_df,
    "train"
)

X_val, y_val, meta_val = build_features(
    val_df,
    "validation"
)

X_test, y_test, meta_test = build_features(
    test_df,
    "test"
)


### 3.5 Inspect a representative feature map


In [ ]:
sample_index = 0

plt.figure(figsize=(10, 4))

plt.imshow(
    X_train[sample_index, :, :, 0],
    origin="lower",
    aspect="auto"
)

sample_label = idx_to_class[int(y_train[sample_index])]

plt.title(f"Training example: {sample_label}")
plt.xlabel("Time frame")
plt.ylabel("Mel-frequency band")
plt.colorbar(label="Normalized Log-Mel value")

plt.tight_layout()
plt.show()

print("Label ID:", y_train[sample_index])
print("Label name:", sample_label)
print("Single-example shape:", X_train[sample_index].shape)


## 4. Feature augmentation and preprocessing checks

### 4.1 Optional synthetic noise augmentation


In [ ]:
def add_white_noise_at_snr(signal, snr_db, rng):
    signal_power = np.mean(signal**2)

    if signal_power < 1e-12:
        return signal.copy()

    noise = rng.standard_normal(len(signal)).astype(np.float32)

    noise_power = np.mean(noise**2)

    desired_noise_power = signal_power / (10**(snr_db / 10))

    noise = noise * np.sqrt(
        desired_noise_power / (noise_power + 1e-12)
    )

    return (signal + noise).astype(np.float32)


### 4.2 Feature extraction helper


In [ ]:
def extract_logmel(segment):

    mel_power = librosa.feature.melspectrogram(
        y=segment,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    log_mel = librosa.power_to_db(
        mel_power,
        ref=np.max
    )

    log_mel = np.clip(log_mel, -80.0, 0.0)

    log_mel = (log_mel + 80.0) / 80.0

    return log_mel.astype(np.float32)


In [ ]:
def build_features(frame, split_name, noise_snr_db=None, random_seed=42):

    # Create a random-number generator for reproducible noise experiments
    rng = np.random.default_rng(random_seed)

    # Lists to store the features, labels, and segment information
    features = []
    labels = []
    metadata = []

    # Counters for checking preprocessing
    invalid_segments = 0
    cropped_segments = 0

    # Group annotation rows by audio file so each recording is loaded once
    for recording_number, (wav_path, group) in enumerate(
        frame.groupby("wav_path", sort=False), start=1
    ):

        # Load the recording and resample it to the target sampling rate
        audio, sr = librosa.load(
            wav_path,
            sr=TARGET_SR,
            mono=True
        )

        # Process each annotated segment in this recording
        for _, row in group.iterrows():

            # Convert annotation timestamps from seconds to sample indices
            start_idx = int(np.floor(row["start"] * sr))
            end_idx = int(np.ceil(row["end"] * sr))

            # Keep the indices within the available recording boundaries
            start_idx = max(0, start_idx)
            end_idx = min(len(audio), end_idx)

            # Skip intervals that do not contain valid audio
            if end_idx <= start_idx:
                invalid_segments += 1
                continue

            # Extract the annotated portion of the recording
            segment = audio[start_idx:end_idx]

            # Count segments that are longer than our target clip length
            if len(segment) > TARGET_SAMPLES:
                cropped_segments += 1

            # Crop or pad the segment to the fixed length
            segment = fit_audio_length(
                segment,
                TARGET_SAMPLES
            )

            # Record the noise level used for this segment
            applied_snr = None

            # Add synthetic white noise only when requested
            if noise_snr_db is not None:

                # If several SNR values are provided, select one
                if isinstance(
                    noise_snr_db,
                    (list, tuple, np.ndarray)
                ):
                    applied_snr = float(
                        rng.choice(noise_snr_db)
                    )

                # Otherwise, use the single specified SNR
                else:
                    applied_snr = float(noise_snr_db)

                # Add noise at the selected signal-to-noise ratio
                segment = add_white_noise_at_snr(
                    segment,
                    applied_snr,
                    rng
                )

            # Convert the audio segment into log-Mel features
            log_mel = extract_logmel(segment)

            # Add a channel dimension for the CNN
            log_mel = log_mel[..., np.newaxis]

            # Read the sound-event label from the annotation
            label = str(row["label"])

            if label not in class_to_idx:
                raise ValueError(
                    f"Unknown label '{label}' in {wav_path}"
                )

            # Convert the class name into an integer ID
            label_id = class_to_idx[label]

            # Store the feature array and corresponding label
            features.append(log_mel)
            labels.append(label_id)

            # Store information that lets us trace each segment
            metadata.append({
                "wav_path": str(wav_path),
                "recording": row["recording"],
                "patient": row["patient"],
                "start": float(row["start"]),
                "end": float(row["end"]),
                "label": label,
                "label_id": label_id,
                "noise_snr_db": applied_snr
            })

        # Display progress after every 100 recordings
        if recording_number % 100 == 0:
            print(
                f"{split_name}: processed "
                f"{recording_number} recordings"
            )

    # Make sure at least one valid feature was produced
    if not features:
        raise ValueError(
            f"No valid features were generated for {split_name}"
        )

    # Convert the lists into NumPy arrays
    X = np.stack(features).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)

    # Convert the metadata list into a Pandas DataFrame
    metadata_df = pd.DataFrame(metadata)

    # Display a summary of the completed preprocessing
    print(f"\n{split_name} complete")
    print("Features shape:", X.shape)
    print("Labels shape:", y.shape)
    print("Invalid segments skipped:", invalid_segments)
    print("Segments center-cropped:", cropped_segments)
    print(
        "Feature memory (MB):",
        round(X.nbytes / (1024 ** 2), 2)
    )

    # Return the features, labels, and metadata
    return X, y, metadata_df


### 4.3 Save features and metadata


In [ ]:
import numpy as np
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save features and labels
np.save(processed_dir / "X_train.npy", X_train)
np.save(processed_dir / "y_train.npy", y_train)

np.save(processed_dir / "X_val.npy", X_val)
np.save(processed_dir / "y_val.npy", y_val)

np.save(processed_dir / "X_test.npy", X_test)
np.save(processed_dir / "y_test.npy", y_test)

# Save metadata
meta_train.to_csv(processed_dir / "meta_train.csv", index=False)
meta_val.to_csv(processed_dir / "meta_val.csv", index=False)
meta_test.to_csv(processed_dir / "meta_test.csv", index=False)

print("All processed datasets saved successfully.")


### 4.4 Inspect class balance


In [ ]:
for label_id, label_name in idx_to_class.items():

    count = np.sum(y_train == label_id)

    print(label_name, ":", count)


### 4.5 Calculate class weights


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
# Get the numerical IDs of our four classes
classes = np.arange(len(CLASSES))
# Check that every class exists in the training data
present_classes = np.unique(y_train)

if len(present_classes) != len(CLASSES):
    raise ValueError("At least one class is missing from training data.")
# Calculate a weight for each class
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
# Create a dictionary connecting class IDs to weights
class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, weights)
}
# Display the weights with their class names
for class_id, weight in class_weights.items():
    print(idx_to_class[class_id], ":", round(weight, 3))


## 5. Compact CNN baseline

### 5.1 Define the compact CNN baseline


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential(name="lung_sound_cnn")

# Specify the shape of one input example
model.add(
    layers.Input(shape=X_train.shape[1:])
)

# Display the expected input dimensions
print("Input shape per example:", X_train.shape[1:])
print("Number of training examples:", X_train.shape[0])


In [ ]:
# Learn 16 patterns from the spectrogram
model.add(
    layers.Conv2D(
        filters=16,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )
)
# Normalize the output of the convolution
model.add(layers.BatchNormalization())
# Apply the ReLU activation function
model.add(layers.Activation("relu"))
# Reduce the dimensions of the feature maps
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
# Help prevent overfitting
model.add(layers.Dropout(0.15))
# Display the model architecture so far
model.summary()


In [ ]:
# Learn 32 additional patterns from the previous feature maps
model.add(
    layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )
)
# Normalize the convolution output
model.add(layers.BatchNormalization())
# Apply the ReLU activation function
model.add(layers.Activation("relu"))
# Reduce the feature-map dimensions
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
# Apply dropout to help reduce overfitting
model.add(layers.Dropout(0.20))
# Inspect the updated network
model.summary()


In [ ]:
# Learn 64 filters from the previous feature maps
model.add(
    layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    )
)
# Normalize the activations
model.add(layers.BatchNormalization())
# Apply the ReLU activation function
model.add(layers.Activation("relu"))
# Check the architecture so far
model.summary()


In [ ]:
# Convert feature maps into a compact feature vector
model.add(layers.GlobalAveragePooling2D())
# Apply dropout to help reduce overfitting
model.add(layers.Dropout(0.25))
# Predict probabilities for our four classes
model.add(
    layers.Dense(
        units=len(CLASSES),
        activation="softmax"
    )
)
model.summary()


### 5.2 Compile the baseline


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")


### 5.3 Training callbacks and checkpointing


In [ ]:
from pathlib import Path
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# Create the folder for saving trained models
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Choose where to save the best model
checkpoint_path = models_dir / "best_logmel_cnn.keras"

# Stop training if validation loss stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

# Save the model whenever validation loss improves
model_checkpoint = ModelCheckpoint(
    filepath=str(checkpoint_path),
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

# Reduce the learning rate when validation loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

# Collect all three callbacks into one list
callbacks = [
    early_stopping,
    model_checkpoint,
    reduce_lr
]

print("Callbacks prepared successfully.")
print("Model checkpoint:", checkpoint_path.resolve())


### 5.4 Train the baseline model


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

print("Training complete.")


### 5.5 Small-sample learning diagnostic


In [ ]:
import numpy as np

# Create a reproducible random-number generator
rng = np.random.default_rng(42)

# Store the indices selected from each class
debug_indices = []

# Select 8 examples from each of the four classes
for class_id in range(len(CLASSES)):

    # Find all training examples belonging to this class
    class_indices = np.flatnonzero(y_train == class_id)

    # Randomly select 8 examples without replacement
    chosen_indices = rng.choice(
        class_indices,
        size=8,
        replace=False
    )

    # Add these indices to our selection
    debug_indices.extend(chosen_indices.tolist())

# Shuffle the selected indices
debug_indices = np.array(debug_indices)
rng.shuffle(debug_indices)

# Create the small diagnostic dataset
X_debug = X_train[debug_indices]
y_debug = y_train[debug_indices]

print("Debug features shape:", X_debug.shape)
print("Debug labels shape:", y_debug.shape)

for class_id, class_name in idx_to_class.items():
    print(
        class_name,
        ":",
        np.sum(y_debug == class_id)
    )


In [ ]:
debug_model = keras.models.clone_model(model)

# Configure its learning process
debug_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Diagnostic model created.")
print("Parameters:", debug_model.count_params())


In [ ]:
debug_history = debug_model.fit(
    X_debug,
    y_debug,
    epochs=30,
    batch_size=8,
    shuffle=True,
    verbose=0
)

# Predict the classes after training
debug_probabilities = debug_model.predict(
    X_debug,
    batch_size=8,
    verbose=0
)

debug_predictions = np.argmax(
    debug_probabilities,
    axis=1
)

# Calculate accuracy on the same 32 examples
debug_accuracy = np.mean(
    debug_predictions == y_debug
)

print(
    "Training accuracy in epoch 1:",
    round(debug_history.history["accuracy"][0], 3)
)

print(
    "Training accuracy in epoch 30:",
    round(debug_history.history["accuracy"][-1], 3)
)

print(
    "Final accuracy with inference predictions:",
    round(debug_accuracy, 3)
)


### 5.6 Normalization diagnostic


In [ ]:
# Create a fresh CNN without Batch Normalization or Dropout
debug_simple_model = keras.Sequential([
    layers.Input(shape=X_debug.shape[1:]),

    # First convolution block
    layers.Conv2D(
        16,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Second convolution block
    layers.Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Third convolution block
    layers.Conv2D(
        64,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),

    # Reduce feature maps to a vector
    layers.GlobalAveragePooling2D(),

    # Predict one of four classes
    layers.Dense(
        len(CLASSES),
        activation="softmax"
    )
])

# Configure learning
debug_simple_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Simple diagnostic model created.")
print("Total parameters:", debug_simple_model.count_params())


In [ ]:
# Train only on the small, balanced diagnostic dataset
simple_history = debug_simple_model.fit(
    X_debug,
    y_debug,
    epochs=100,
    batch_size=8,
    shuffle=True,
    verbose=0
)

# Evaluate the trained model in inference mode
simple_loss, simple_accuracy = debug_simple_model.evaluate(
    X_debug,
    y_debug,
    verbose=0
)

print(
    "Accuracy in epoch 1:",
    round(simple_history.history["accuracy"][0], 3)
)

print(
    "Accuracy in epoch 100:",
    round(simple_history.history["accuracy"][-1], 3)
)

print(
    "Final evaluation accuracy:",
    round(simple_accuracy, 3)
)


In [ ]:
# Create a fresh neural network for the diagnostic
debug_mlp = keras.Sequential([
    layers.Input(shape=X_debug.shape[1:]),

    # Turn the 2D spectrogram into a 1D vector
    layers.Flatten(),

    # Learn patterns from the flattened features
    layers.Dense(64, activation="relu"),

    # Output probabilities for the four classes
    layers.Dense(len(CLASSES), activation="softmax")
])

# Configure the learning process
debug_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Parameters:", debug_mlp.count_params())


In [ ]:
# Train only on the diagnostic examples
mlp_history = debug_mlp.fit(
    X_debug,
    y_debug,
    epochs=100,
    batch_size=8,
    shuffle=True,
    verbose=0
)

# Evaluate the final model on those same examples
mlp_loss, mlp_accuracy = debug_mlp.evaluate(
    X_debug,
    y_debug,
    verbose=0
)

print(
    "Accuracy in epoch 1:",
    round(mlp_history.history["accuracy"][0], 3)
)

print(
    "Accuracy in epoch 100:",
    round(mlp_history.history["accuracy"][-1], 3)
)

print("Final training accuracy:", round(mlp_accuracy, 3))


In [ ]:
# Create a new CNN for our diagnostic
debug_cnn_flatten = keras.Sequential([
    layers.Input(shape=X_debug.shape[1:]),

    # First convolution block
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Second convolution block
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Third convolution block
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),

    # Preserve the spatial positions of learned features
    layers.Flatten(),

    # Learn how the extracted features relate to each class
    layers.Dense(64, activation="relu"),

    # Predict the four sound-event classes
    layers.Dense(len(CLASSES), activation="softmax")
])

# Configure training
debug_cnn_flatten.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

debug_cnn_flatten.summary()


In [ ]:
# Train on the balanced diagnostic subset
flatten_history = debug_cnn_flatten.fit(
    X_debug,
    y_debug,
    epochs=50,
    batch_size=8,
    shuffle=True,
    verbose=0
)

# Evaluate on those same training examples
flatten_loss, flatten_accuracy = debug_cnn_flatten.evaluate(
    X_debug,
    y_debug,
    verbose=0
)

print(
    "Accuracy in epoch 1:",
    round(flatten_history.history["accuracy"][0], 3)
)

print(
    "Accuracy in epoch 50:",
    round(flatten_history.history["accuracy"][-1], 3)
)

print(
    "Final training accuracy:",
    round(flatten_accuracy, 3)
)


In [ ]:
debug_compact_model = keras.Sequential([
    layers.Input(shape=X_debug.shape[1:]),

    # First convolution block
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Second convolution block
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Third convolution block
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),

    # Preserve coarse time-frequency locations
    layers.AveragePooling2D(pool_size=(4, 4)),

    # Convert the pooled feature maps into a vector
    layers.Flatten(),

    # Learn a compact representation for classification
    layers.Dense(32, activation="relu"),

    # Predict probabilities for the four classes
    layers.Dense(len(CLASSES), activation="softmax")
])

debug_compact_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Parameters:", debug_compact_model.count_params())


In [ ]:
compact_history = debug_compact_model.fit(
    X_debug,
    y_debug,
    epochs=100,
    batch_size=8,
    shuffle=True,
    verbose=0
)

compact_loss, compact_accuracy = debug_compact_model.evaluate(
    X_debug,
    y_debug,
    verbose=0
)

print(
    "Accuracy in epoch 1:",
    round(compact_history.history["accuracy"][0], 3)
)

print(
    "Accuracy in epoch 100:",
    round(compact_history.history["accuracy"][-1], 3)
)

print(
    "Final training accuracy:",
    round(compact_accuracy, 3)
)


In [ ]:
# Copy the compact CNN architecture, not its trained weights
compact_model = keras.models.clone_model(debug_compact_model)

# Configure the training process
compact_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fresh compact CNN created.")
print("Total parameters:", compact_model.count_params())

compact_model.summary()


### 5.7 Train the compact CNN on the full development split


In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from pathlib import Path

# Create the folder for saving models
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Use a separate checkpoint for the compact CNN
compact_checkpoint = models_dir / "best_compact_logmel_cnn.keras"

# Stop when validation loss stops improving
compact_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

# Save the best model based on validation loss
compact_model_checkpoint = ModelCheckpoint(
    filepath=str(compact_checkpoint),
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

# Reduce the learning rate when validation loss plateaus
compact_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

# Combine the callbacks into one list
compact_callbacks = [
    compact_early_stopping,
    compact_model_checkpoint,
    compact_reduce_lr
]

print("Compact CNN callbacks prepared.")
print("Checkpoint path:", compact_checkpoint.resolve())


In [ ]:
compact_history = compact_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=compact_callbacks,
    shuffle=True,
    verbose=1
)

print("Compact CNN training complete.")


### 5.8 Evaluate the compact CNN


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
# Evaluate the compact model on validation data
val_loss, val_accuracy = compact_model.evaluate(
    X_val,
    y_val,
    batch_size=32,
    verbose=0
)

print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_accuracy, 4))
# Generate class probabilities
val_probabilities_compact = compact_model.predict(
    X_val,
    batch_size=32,
    verbose=0
)
# Select the class with the highest probability
val_predictions_compact = np.argmax(
    val_probabilities_compact,
    axis=1
)
# Calculate accuracy and macro-F1
val_accuracy_check = accuracy_score(
    y_val,
    val_predictions_compact
)

val_macro_f1 = f1_score(
    y_val,
    val_predictions_compact,
    average="macro",
    zero_division=0
)

print("\nAccuracy from predictions:", round(val_accuracy_check, 4))
print("Validation macro-F1:", round(val_macro_f1, 4))
# Count predictions for each class
print("\nPredicted class counts:")

for class_id, class_name in idx_to_class.items():
    count = np.sum(val_predictions_compact == class_id)
    print(f"{class_name}: {count}")
# Show the confusion matrix
print("\nValidation confusion matrix:")

print(
    confusion_matrix(
        y_val,
        val_predictions_compact,
        labels=np.arange(len(CLASSES))
    )
)
# Show detailed performance for each class
print("\nValidation classification report:")

print(
    classification_report(
        y_val,
        val_predictions_compact,
        labels=np.arange(len(CLASSES)),
        target_names=CLASSES,
        zero_division=0
    )
)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import numpy as np
# Generate predictions on the training set
train_probabilities_compact = compact_model.predict(
    X_train,
    batch_size=32,
    verbose=0
)
# Convert the probabilities into class IDs
train_predictions_compact = np.argmax(
    train_probabilities_compact,
    axis=1
)
# Calculate training accuracy
train_accuracy_compact = accuracy_score(
    y_train,
    train_predictions_compact
)
# Calculate training macro-F1
train_macro_f1_compact = f1_score(
    y_train,
    train_predictions_compact,
    average="macro",
    zero_division=0
)

print("Training accuracy:", round(train_accuracy_compact, 4))
print("Training macro-F1:", round(train_macro_f1_compact, 4))
# Display performance for each class
print("\nTraining classification report:")

print(
    classification_report(
        y_train,
        train_predictions_compact,
        labels=np.arange(len(CLASSES)),
        target_names=CLASSES,
        zero_division=0
    )
)
# Count predictions for each class
print("\nTraining predicted class counts:")

for class_id, class_name in idx_to_class.items():
    count = np.sum(train_predictions_compact == class_id)
    print(f"{class_name}: {count}")


## 6. Split audit and robustness diagnostics

### 6.1 Check split sizes and patient coverage


In [ ]:
for split_name, metadata in [
    ("Training", meta_train),
    ("Validation", meta_val),
    ("Test", meta_test)
]:
    print(f"\n{split_name}")
    print("Segments:", len(metadata))
    print("Unique patients:", metadata["patient"].nunique())
    print("Unique recordings:", metadata["recording"].nunique())


# Convert patient IDs and recording names to strings for comparison
train_patients = set(meta_train["patient"].astype(str))
val_patients = set(meta_val["patient"].astype(str))
test_patients = set(meta_test["patient"].astype(str))

train_recordings = set(meta_train["recording"].astype(str))
val_recordings = set(meta_val["recording"].astype(str))
test_recordings = set(meta_test["recording"].astype(str))


print("\nPatient overlap checks")
print("Train vs validation:", train_patients & val_patients)
print("Train vs test:", train_patients & test_patients)
print("Validation vs test:", val_patients & test_patients)


print("\nRecording overlap checks")
print("Train vs validation:", train_recordings & val_recordings)
print("Train vs test:", train_recordings & test_recordings)
print("Validation vs test:", val_recordings & test_recordings)


patient_ids = meta_val["patient"].astype(str).str.replace(
    r"\.0$", "", regex=True
)

patient_226_mask = patient_ids == "226"

print("\nPatient 226 validation segments:")
print(
    meta_val.loc[patient_226_mask, "label"]
    .value_counts()
    .reindex(CLASSES, fill_value=0)
)

print("\nOther validation patients:")
print(
    meta_val.loc[~patient_226_mask, "label"]
    .value_counts()
    .reindex(CLASSES, fill_value=0)
)


### 6.2 Patient-grouped cross-validation diagnostic


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Divide the eligible training pool into six patient-grouped folds
splitter = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

# Select one fold for validation
train_indices, val_indices = next(
    splitter.split(
        train_pool,
        y=train_pool["label"],
        groups=train_pool["patient"]
    )
)

# Create candidate training and validation DataFrames
candidate_train_df = train_pool.iloc[train_indices].copy()
candidate_val_other = train_pool.iloc[val_indices].copy()

# Keep patient 226 in validation as before
patient226_rows = val_df[
    val_df["patient"].astype(str) == "226"
].copy()

candidate_val_df = pd.concat(
    [candidate_val_other, patient226_rows],
    ignore_index=True
)

print("Candidate training segments:", len(candidate_train_df))
print("Candidate validation segments:", len(candidate_val_df))

print("\nCandidate validation class coverage:")

for class_name in CLASSES:
    class_rows = candidate_val_df[
        candidate_val_df["label"] == class_name
    ]

    print(
        f"{class_name}: "
        f"{len(class_rows)} segments, "
        f"{class_rows['patient'].nunique()} patients, "
        f"{class_rows['recording'].nunique()} recordings"
    )

candidate_train_patients = set(
    candidate_train_df["patient"].astype(str)
)

candidate_val_patients = set(
    candidate_val_df["patient"].astype(str)
)

print(
    "\nPatient overlap:",
    candidate_train_patients & candidate_val_patients
)


### 6.3 Train the fold-specific CNN


In [ ]:
# Process the new training split
X_train_new, y_train_new, meta_train_new = build_features(
    candidate_train_df,
    "train_fold3"
)

# Process the new validation split
X_val_new, y_val_new, meta_val_new = build_features(
    candidate_val_df,
    "validation_fold3"
)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
# Define the four numerical class IDs
classes = np.arange(len(CLASSES))
# Verify that all four classes exist in training
present_classes = np.unique(y_train_new)

if len(present_classes) != len(CLASSES):
    raise ValueError("At least one class is missing from training.")
# Calculate balanced weights using the new training labels
weights_new = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_new
)
# Map each class ID to its weight
class_weights_new = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, weights_new)
}
# Display the results
print("Updated class weights:")

for class_id, weight in class_weights_new.items():
    print(
        idx_to_class[class_id],
        ":",
        round(weight, 3)
    )


In [ ]:
# Create a fresh compact CNN
fold3_model = keras.Sequential([
    layers.Input(shape=X_train_new.shape[1:]),

    # Convolution block 1
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 2
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 3
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),

    # Reduce the feature maps while preserving spatial information
    layers.AveragePooling2D((4, 4)),
    layers.Flatten(),

    # Learn a compact representation
    layers.Dense(32, activation="relu"),

    # Output probabilities for the four classes
    layers.Dense(len(CLASSES), activation="softmax")
])

# Configure the learning process
fold3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Parameters:", fold3_model.count_params())
fold3_model.summary()


In [ ]:
# Train the fresh CNN on the new training split
fold3_history = fold3_model.fit(
    X_train_new,
    y_train_new,

    # Evaluate on the new validation split after each epoch
    validation_data=(X_val_new, y_val_new),

    # Maximum number of complete passes through the training data
    epochs=30,

    # Process 32 examples per batch
    batch_size=32,

    # Account for the imbalance in the new training split
    class_weight=class_weights_new,

    # Enable early stopping, checkpointing and learning-rate reduction
    callbacks=fold3_callbacks,

    # Shuffle the training examples between epochs
    shuffle=True,

    # Display training progress
    verbose=1
)

print("Fold 3 training complete.")


### 7.1 Record exclusions used to preserve separation


In [ ]:
from pathlib import Path

audit_path = Path("../data/processed/excluded_recordings_audit.csv")
audit_path.parent.mkdir(parents=True, exist_ok=True)

missing_df.to_csv(audit_path, index=False)

print("Exclusion audit saved to:", audit_path.resolve())
print("Excluded recordings:", len(missing_df))
print("Affected patients:", missing_df["patient"].nunique())


## 8. Baselines and controlled model comparisons

### 8.1 Simple baseline classifiers


In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Calculate one duration value for every training segment
duration_train = (
    meta_train_new["end"] - meta_train_new["start"]
).to_numpy().reshape(-1, 1)

# Calculate one duration value for every validation segment
duration_val = (
    meta_val_new["end"] - meta_val_new["start"]
).to_numpy().reshape(-1, 1)

print("Training duration shape:", duration_train.shape)
print("Validation duration shape:", duration_val.shape)


### 8.2 Segment-duration baseline


In [ ]:
# Create a baseline model using only segment duration
duration_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

# Train using durations and their corresponding class labels
duration_model.fit(duration_train, y_train_new)

# Predict labels from duration alone
duration_predictions = duration_model.predict(duration_val)

# Evaluate predictions on the validation set
print(
    "Duration-only validation accuracy:",
    round(accuracy_score(y_val_new, duration_predictions), 4)
)

print(
    "Duration-only validation macro-F1:",
    round(
        f1_score(
            y_val_new,
            duration_predictions,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print("\nDuration-only classification report:")
print(
    classification_report(
        y_val_new,
        duration_predictions,
        labels=np.arange(len(CLASSES)),
        target_names=CLASSES,
        zero_division=0
    )
)


### 8.3 Always-normal reference


In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)
# Create a baseline that always predicts the most common training class
majority_model = DummyClassifier(strategy="most_frequent")

# Fit the baseline using the new training labels
majority_model.fit(
    np.zeros((len(y_train_new), 1)),
    y_train_new
)
# Predict the validation labels
majority_predictions = majority_model.predict(
    np.zeros((len(y_val_new), 1))
)
# Compare the three approaches
models_to_compare = {
    "Majority baseline": majority_predictions,
    "Duration-only model": duration_predictions,
    "Compact CNN": val_predictions_fold3
}

for name, predictions in models_to_compare.items():

    accuracy = accuracy_score(y_val_new, predictions)

    macro_f1 = f1_score(
        y_val_new,
        predictions,
        average="macro",
        zero_division=0
    )

    balanced_acc = balanced_accuracy_score(
        y_val_new,
        predictions
    )

    print(f"\n{name}")
    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Macro-F1:          {macro_f1:.4f}")
    print(f"Balanced accuracy: {balanced_acc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use a fixed random seed so the selected examples are reproducible
rng = np.random.default_rng(42)

# Create a figure with four rows and two columns
fig, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(12, 14)
)

for class_id, class_name in idx_to_class.items():

    # Find training examples belonging to this class
    train_indices = np.flatnonzero(
        y_train_new == class_id
    )

    # Find validation examples belonging to this class
    val_indices = np.flatnonzero(
        y_val_new == class_id
    )

    # Select one example from each split
    train_idx = rng.choice(train_indices)
    val_idx = rng.choice(val_indices)

    # Extract the 2D log-Mel arrays
    train_spec = X_train_new[train_idx, :, :, 0]
    val_spec = X_val_new[val_idx, :, :, 0]

    # Plot the training example
    axes[class_id, 0].imshow(
        train_spec,
        origin="lower",
        aspect="auto",
        vmin=0,
        vmax=1
    )

    axes[class_id, 0].set_title(
        f"Training: {class_name}"
    )
    axes[class_id, 0].set_xlabel("Time frame")
    axes[class_id, 0].set_ylabel("Mel band")

    # Plot the validation example
    axes[class_id, 1].imshow(
        val_spec,
        origin="lower",
        aspect="auto",
        vmin=0,
        vmax=1
    )

    axes[class_id, 1].set_title(
        f"Validation: {class_name}"
    )
    axes[class_id, 1].set_xlabel("Time frame")
    axes[class_id, 1].set_ylabel("Mel band")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

def resize_logmel_time(log_mel, target_frames=157):

    source_frames = log_mel.shape[1]

    # If it already has the required number of frames, return it
    if source_frames == target_frames:
        return log_mel.astype(np.float32)

    # Create evenly spaced time positions for the original spectrogram
    source_positions = np.linspace(
        0.0, 1.0, source_frames
    )

    # Create evenly spaced positions for the resized spectrogram
    target_positions = np.linspace(
        0.0, 1.0, target_frames
    )

    # Allocate an output array with 64 Mel bands and target_frames
    resized = np.empty(
        (log_mel.shape[0], target_frames),
        dtype=np.float32
    )

    # Resize each Mel band's values along the time dimension
    for band_index in range(log_mel.shape[0]):

        resized[band_index] = np.interp(
            target_positions,
            source_positions,
            log_mel[band_index]
        )

    # Return the resized spectrogram
    return resized


## 9. Alternative time and feature representations

The next experiments test whether the fixed time dimension or the feature representation is limiting the baseline models. The comparison includes time resizing before moving to frame-level padding and multi-channel inputs.


In [ ]:
import librosa
import numpy as np

# Select one annotated training segment
sample_row = meta_train_new.iloc[0]

# Load its original recording at the target sampling rate
audio, sr = librosa.load(
    sample_row["wav_path"],
    sr=TARGET_SR,
    mono=True
)

# Convert the annotation timestamps into sample indices
start_idx = int(np.floor(sample_row["start"] * sr))
end_idx = int(np.ceil(sample_row["end"] * sr))

# Keep the indices within the recording boundaries
start_idx = max(0, start_idx)
end_idx = min(len(audio), end_idx)

# Extract the actual annotated audio
segment = audio[start_idx:end_idx]

# Centre-crop only if the segment exceeds five seconds
if len(segment) > TARGET_SAMPLES:
    segment = fit_audio_length(
        segment,
        TARGET_SAMPLES
    )

# Extract features without padding the waveform to five seconds
log_mel_variable = extract_logmel(segment)

# Resize the time dimension to 157 frames
log_mel_resized = resize_logmel_time(
    log_mel_variable,
    target_frames=X_train_new.shape[2]
)

# Display the results
print("Original segment duration:", round(len(segment) / sr, 3), "seconds")
print("Before resizing:", log_mel_variable.shape)
print("After resizing:", log_mel_resized.shape)
print("Minimum value:", round(float(log_mel_resized.min()), 4))
print("Maximum value:", round(float(log_mel_resized.max()), 4))
print("Data type:", log_mel_resized.dtype)


### 9.2 Build time-resized features


In [ ]:
def build_features_resized(
    frame,
    split_name,
    noise_snr_db=None,
    random_seed=42,
    target_frames=157
):
# Create a reproducible random-number generator
    rng = np.random.default_rng(random_seed)
# Create lists for features, labels, and metadata
    features = []
    labels = []
    metadata = []
# Create counters for data-quality checks
    invalid_segments = 0
    cropped_segments = 0
# Group annotation rows by recording
    for recording_number, (wav_path, group) in enumerate(
        frame.groupby("wav_path", sort=False),
        start=1
    ):
# Load each recording once at the target sample rate
        audio, sr = librosa.load(
            wav_path,
            sr=TARGET_SR,
            mono=True
        )
# Process each annotated interval in the recording
        for _, row in group.iterrows():

            # Convert timestamps from seconds to sample indices
            start_idx = int(np.floor(row["start"] * sr))
            end_idx = int(np.ceil(row["end"] * sr))

            # Keep the indices within the recording boundaries
            start_idx = max(0, start_idx)
            end_idx = min(len(audio), end_idx)

            # Skip invalid intervals
            if end_idx <= start_idx:
                invalid_segments += 1
                continue

            # Extract the annotated audio segment
            segment = audio[start_idx:end_idx]

            # Crop only if the segment is longer than five seconds
            if len(segment) > TARGET_SAMPLES:
                cropped_segments += 1
                segment = fit_audio_length(
                    segment,
                    TARGET_SAMPLES
                )

            # Record the noise setting for this segment
            applied_snr = None

            # Optionally add simulated white noise
            if noise_snr_db is not None:

                if isinstance(
                    noise_snr_db,
                    (list, tuple, np.ndarray)
                ):
                    applied_snr = float(
                        rng.choice(noise_snr_db)
                    )
                else:
                    applied_snr = float(noise_snr_db)

                segment = add_white_noise_at_snr(
                    segment,
                    applied_snr,
                    rng
                )

            # Extract log-Mel features without five-second waveform padding
            log_mel = extract_logmel(segment)

            # Resize the time dimension to a fixed number of frames
            log_mel = resize_logmel_time(
                log_mel,
                target_frames=target_frames
            )

            # Add a channel dimension for the CNN
            log_mel = log_mel[..., np.newaxis]

            # Get the annotation label
            label = str(row["label"])

            if label not in class_to_idx:
                raise ValueError(
                    f"Unknown label '{label}' in {wav_path}"
                )

            # Convert the label into its numerical ID
            label_id = class_to_idx[label]

            # Store the spectrogram and its corresponding label
            features.append(log_mel)
            labels.append(label_id)

            # Store information about the original audio interval
            metadata.append({
                "wav_path": str(wav_path),
                "recording": row["recording"],
                "patient": row["patient"],
                "start": float(row["start"]),
                "end": float(row["end"]),
                "label": label,
                "label_id": label_id,
                "noise_snr_db": applied_snr
            })

        # Display progress every 100 recordings
        if recording_number % 100 == 0:
            print(
                f"{split_name}: processed "
                f"{recording_number} recordings"
            )

    # Make sure the function produced at least one feature
    if not features:
        raise ValueError(
            f"No valid features generated for {split_name}"
        )

    # Convert the feature list into a NumPy array
    X = np.stack(features).astype(np.float32)

    # Convert the label list into an integer NumPy array
    y = np.asarray(labels, dtype=np.int64)

    # Convert metadata into a DataFrame
    metadata_df = pd.DataFrame(metadata)

    print(f"\n{split_name} complete")
    print("Features shape:", X.shape)
    print("Labels shape:", y.shape)
    print("Invalid segments skipped:", invalid_segments)
    print("Segments center-cropped:", cropped_segments)
    print(
        "Feature memory (MB):",
        round(X.nbytes / (1024 ** 2), 2)
    )

    # Return the features, labels, and metadata
    return X, y, metadata_df


In [ ]:
# Select three examples per class from the new training split
small_train_df = (
    candidate_train_df
    .groupby("label", group_keys=False)
    .head(3)
    .copy()
)

# Select three examples per class from the new validation split
small_val_df = (
    candidate_val_df
    .groupby("label", group_keys=False)
    .head(3)
    .copy()
)

print("Small training sample:", len(small_train_df))
print("Small validation sample:", len(small_val_df))

# Process the small training sample
X_small_train, y_small_train, meta_small_train = (
    build_features_resized(
        small_train_df,
        "small_train"
    )
)

# Process the small validation sample
X_small_val, y_small_val, meta_small_val = (
    build_features_resized(
        small_val_df,
        "small_validation"
    )
)


In [ ]:
# Process all training segments using the new preprocessing method
X_train_resized, y_train_resized, meta_train_resized = (
    build_features_resized(
        candidate_train_df,
        "train_resized"
    )
)

# Process all validation segments using the same method
X_val_resized, y_val_resized, meta_val_resized = (
    build_features_resized(
        candidate_val_df,
        "validation_resized"
    )
)

print("\nFull resized dataset processing complete.")


### 9.3 Prepare class weighting for the resized model


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Get the four numerical class IDs
classes = np.arange(len(CLASSES))

present_classes = np.unique(y_train_resized)

if len(present_classes) != len(CLASSES):
    raise ValueError("At least one class is missing from training.")

# Calculate balanced weights from the resized training labels
weights_resized = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_resized
)

# Create the dictionary used by Keras during training
class_weights_resized = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, weights_resized)
}

print("Class weights for resized features:")

for class_id, weight in class_weights_resized.items():
    print(
        f"{idx_to_class[class_id]}: {weight:.3f}"
    )


### 9.4 Define the resized CNN


In [ ]:
# Create a fresh CNN for the resized spectrograms
resized_model = keras.Sequential([
    layers.Input(shape=X_train_resized.shape[1:]),

    # Block 1: learn 16 patterns
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Block 2: learn 32 patterns
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Block 3: learn 64 patterns
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),

    # Preserve coarse time-frequency information
    layers.AveragePooling2D((4, 4)),
    layers.Flatten(),

    # Learn a compact representation
    layers.Dense(32, activation="relu"),

    # Predict probabilities for four classes
    layers.Dense(len(CLASSES), activation="softmax")
])

# Configure the learning process
resized_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display the model's parameter count and architecture
print("Total parameters:", resized_model.count_params())
resized_model.summary()


### 9.5 Training callbacks


In [ ]:
from pathlib import Path
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# Create the folder for saving models
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Choose a separate checkpoint filename
resized_checkpoint = models_dir / "best_resized_logmel_cnn.keras"

# Stop when validation loss stops improving
resized_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

# Save the model with the best validation loss
resized_model_checkpoint = ModelCheckpoint(
    filepath=str(resized_checkpoint),
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

# Reduce the learning rate when progress stalls
resized_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

# Combine all callbacks into one list
resized_callbacks = [
    resized_early_stopping,
    resized_model_checkpoint,
    resized_reduce_lr
]

print("Resized-feature model callbacks prepared.")
print("Checkpoint:", resized_checkpoint.resolve())


### 9.6 Train the resized CNN


In [ ]:
# Train the new CNN using resized training features
resized_history = resized_model.fit(
    X_train_resized,
    y_train_resized,

    # Monitor performance on the validation set
    validation_data=(X_val_resized, y_val_resized),

    # Maximum number of epochs
    epochs=30,

    # Number of examples processed per batch
    batch_size=32,

    # Account for the imbalance between classes
    class_weight=class_weights_resized,

    # Use our callbacks
    callbacks=resized_callbacks,

    # Shuffle training examples between epochs
    shuffle=True,

    # Display progress during training
    verbose=1
)

print("Resized-feature CNN training complete.")


### 9.7 Evaluate the resized CNN


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
# Evaluate the model on resized validation features
val_loss_resized, val_acc_resized = resized_model.evaluate(
    X_val_resized,
    y_val_resized,
    batch_size=32,
    verbose=0
)

print("Validation loss:", round(val_loss_resized, 4))
print("Validation accuracy:", round(val_acc_resized, 4))
# Obtain the predicted probabilities for each class
val_probabilities_resized = resized_model.predict(
    X_val_resized,
    batch_size=32,
    verbose=0
)
# Choose the class with the highest probability
val_predictions_resized = np.argmax(
    val_probabilities_resized,
    axis=1
)
# Calculate accuracy and macro-F1
print(
    "\nAccuracy:",
    round(accuracy_score(
        y_val_resized,
        val_predictions_resized
    ), 4)
)

print(
    "Macro-F1:",
    round(f1_score(
        y_val_resized,
        val_predictions_resized,
        average="macro",
        zero_division=0
    ), 4)
)
# Count predictions for every class
print("\nPredicted class counts:")

for class_id, class_name in idx_to_class.items():
    print(
        f"{class_name}:",
        int(np.sum(val_predictions_resized == class_id))
    )
# Display the confusion matrix
print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_val_resized,
        val_predictions_resized,
        labels=np.arange(len(CLASSES))
    )
)
# Display class-wise performance
print("\nClassification report:")

print(
    classification_report(
        y_val_resized,
        val_predictions_resized,
        labels=np.arange(len(CLASSES)),
        target_names=CLASSES,
        zero_division=0
    )
)


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
# Predict the training examples using the resized-feature model
train_probabilities_resized = resized_model.predict(
    X_train_resized,
    batch_size=32,
    verbose=0
)
# Convert probabilities into predicted class IDs
train_predictions_resized = np.argmax(
    train_probabilities_resized,
    axis=1
)
# Calculate training accuracy and macro-F1
train_accuracy_resized = accuracy_score(
    y_train_resized,
    train_predictions_resized
)

train_macro_f1_resized = f1_score(
    y_train_resized,
    train_predictions_resized,
    average="macro",
    zero_division=0
)

print("Training accuracy:", round(train_accuracy_resized, 4))
print("Training macro-F1:", round(train_macro_f1_resized, 4))
# Display per-class training performance
print("\nTraining classification report:")

print(
    classification_report(
        y_train_resized,
        train_predictions_resized,
        labels=np.arange(len(CLASSES)),
        target_names=CLASSES,
        zero_division=0
    )
)


### 9.8 Frame-length utility


In [ ]:
import numpy as np

def fit_logmel_frames(log_mel, target_frames=157):

    source_frames = log_mel.shape[1]

    # If the spectrogram is too long, crop its time dimension
    if source_frames > target_frames:

        start = (source_frames - target_frames) // 2

        log_mel = log_mel[
            :, start:start + target_frames
        ]

    # If it is too short, add zero-valued frames
    elif source_frames < target_frames:

        total_padding = target_frames - source_frames

        left_padding = total_padding // 2

        right_padding = total_padding - left_padding

        log_mel = np.pad(
            log_mel,
            ((0, 0), (left_padding, right_padding)),
            mode="constant",
            constant_values=0.0
        )

    # Return the result as a float32 array
    return log_mel.astype(np.float32)


In [ ]:
# Select one training segment
sample_row = meta_train_new.iloc[0]

# Load its recording
audio, sr = librosa.load(
    sample_row["wav_path"],
    sr=TARGET_SR,
    mono=True
)

# Convert the annotation times to sample indices
start_idx = int(np.floor(sample_row["start"] * sr))
end_idx = int(np.ceil(sample_row["end"] * sr))

# Keep the indices inside the recording
start_idx = max(0, start_idx)
end_idx = min(len(audio), end_idx)

# Extract the actual annotated segment
segment = audio[start_idx:end_idx]

# Centre-crop only if it exceeds five seconds
if len(segment) > TARGET_SAMPLES:
    segment = fit_audio_length(
        segment,
        TARGET_SAMPLES
    )

# Extract features before adding any padding
log_mel_original = extract_logmel(segment)

# Pad or crop only the spectrogram's time dimension
log_mel_padded = fit_logmel_frames(
    log_mel_original,
    target_frames=157
)

print(
    "Original segment duration:",
    round(len(segment) / sr, 3),
    "seconds"
)

print("Before frame padding:", log_mel_original.shape)
print("After frame padding:", log_mel_padded.shape)

print(
    "Feature range:",
    float(log_mel_padded.min()),
    "to",
    float(log_mel_padded.max())
)

print("Data type:", log_mel_padded.dtype)


## 10. Frame-padded three-channel features

Frame padding is used so that the actual spectrogram is extracted before padding. A validity mask separates real frames from padded frames, and the first and second derivatives are computed only over valid frames.


### 10.2 Build frame-padded features


In [ ]:
def build_features_framepad(
    frame,
    split_name,
    noise_snr_db=None,
    random_seed=42,
    target_frames=157
):
    # Create a random-number generator
    rng = np.random.default_rng(random_seed)

    # Lists for storing features, labels, and metadata
    features = []
    labels = []
    metadata = []

    # Counters for monitoring preprocessing
    invalid_segments = 0
    audio_cropped_segments = 0
    frame_padded_segments = 0
    frame_cropped_segments = 0

    # Group annotation rows by recording
    for recording_number, (wav_path, group) in enumerate(
        frame.groupby("wav_path", sort=False),
        start=1
    ):

        # Load each recording once at the target sample rate
        audio, sr = librosa.load(
            wav_path,
            sr=TARGET_SR,
            mono=True
        )

        # Process each annotated interval
        for _, row in group.iterrows():

            # Convert timestamps from seconds to sample indices
            start_idx = int(np.floor(row["start"] * sr))
            end_idx = int(np.ceil(row["end"] * sr))

            # Keep indices within the recording boundaries
            start_idx = max(0, start_idx)
            end_idx = min(len(audio), end_idx)

            # Skip invalid intervals
            if end_idx <= start_idx:
                invalid_segments += 1
                continue

            # Extract the annotated audio segment
            segment = audio[start_idx:end_idx]

            # Crop long segments, but do not pad short waveforms
            if len(segment) > TARGET_SAMPLES:
                audio_cropped_segments += 1

                segment = fit_audio_length(
                    segment,
                    TARGET_SAMPLES
                )

            # Record the noise setting for this segment
            applied_snr = None

            # Add synthetic white noise only when requested
            if noise_snr_db is not None:

                if isinstance(
                    noise_snr_db,
                    (list, tuple, np.ndarray)
                ):
                    applied_snr = float(
                        rng.choice(noise_snr_db)
                    )
                else:
                    applied_snr = float(noise_snr_db)

                segment = add_white_noise_at_snr(
                    segment,
                    applied_snr,
                    rng
                )

            # Extract features from the actual segment
            log_mel = extract_logmel(segment)

            # Track how many spectrograms need frame adjustment
            source_frames = log_mel.shape[1]

            if source_frames < target_frames:
                frame_padded_segments += 1

            elif source_frames > target_frames:
                frame_cropped_segments += 1

            # Pad or crop the spectrogram's time dimension
            log_mel = fit_logmel_frames(
                log_mel,
                target_frames=target_frames
            )

            # Add the channel dimension required by the CNN
            log_mel = log_mel[..., np.newaxis]

            # Get the label from the annotation
            label = str(row["label"])

            # Ensure the label is in our class mapping
            if label not in class_to_idx:
                raise ValueError(
                    f"Unknown label '{label}' in {wav_path}"
                )

            # Convert the class name to an integer
            label_id = class_to_idx[label]

            # Store the features and their corresponding label
            features.append(log_mel)
            labels.append(label_id)

            # Store information about the original audio segment
            metadata.append({
                "wav_path": str(wav_path),
                "recording": row["recording"],
                "patient": row["patient"],
                "start": float(row["start"]),
                "end": float(row["end"]),
                "label": label,
                "label_id": label_id,
                "noise_snr_db": applied_snr
            })

        # Display progress every 100 recordings
        if recording_number % 100 == 0:
            print(
                f"{split_name}: processed "
                f"{recording_number} recordings"
            )

    # Stop if no valid features were generated
    if not features:
        raise ValueError(
            f"No valid features generated for {split_name}"
        )

    # Convert the feature list into a NumPy array
    X = np.stack(features).astype(np.float32)

    # Convert labels into an integer NumPy array
    y = np.asarray(labels, dtype=np.int64)

    # Convert metadata into a DataFrame
    metadata_df = pd.DataFrame(metadata)

    # Display a summary
    print(f"\n{split_name} complete")
    print("Features shape:", X.shape)
    print("Labels shape:", y.shape)
    print("Invalid segments skipped:", invalid_segments)
    print("Audio segments centre-cropped:", audio_cropped_segments)
    print("Spectrograms frame-padded:", frame_padded_segments)
    print("Spectrograms frame-cropped:", frame_cropped_segments)
    print(
        "Feature memory (MB):",
        round(X.nbytes / (1024 ** 2), 2)
    )

    # Return all three outputs
    return X, y, metadata_df


In [ ]:
# Select three examples per class from training
small_train_df = (
    candidate_train_df
    .groupby("label", group_keys=False)
    .head(3)
    .copy()
)

# Select three examples per class from validation
small_val_df = (
    candidate_val_df
    .groupby("label", group_keys=False)
    .head(3)
    .copy()
)

# Process the small training sample
X_frame_train, y_frame_train, meta_frame_train = (
    build_features_framepad(
        small_train_df,
        "small_frame_train"
    )
)

# Process the small validation sample
X_frame_val, y_frame_val, meta_frame_val = (
    build_features_framepad(
        small_val_df,
        "small_frame_validation"
    )
)


In [ ]:
# Process all training segments using spectrogram frame padding
X_train_framepad, y_train_framepad, meta_train_framepad = (
    build_features_framepad(
        candidate_train_df,
        "train_framepad",
        target_frames=X_train_new.shape[2]
    )
)

# Process all validation segments using the same method
X_val_framepad, y_val_framepad, meta_val_framepad = (
    build_features_framepad(
        candidate_val_df,
        "validation_framepad",
        target_frames=X_train_new.shape[2]
    )
)

print("Full frame-padding preprocessing complete.")


## 11. Weighted and patient-balanced CNNs

These experiments address two sources of bias in the development data: unequal class counts and unequal numbers of cycles contributed by individual patients.


### 11.2 Train the weighted three-channel CNN


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
# Calculate class weights from the training labels only
class_ids = np.unique(y_train_framepad)

weights = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=y_train_framepad
)

class_weights_framepad = {
    int(class_id): float(weight)
    for class_id, weight in zip(class_ids, weights)
}

print("Class weights:")
for class_id, class_name in idx_to_class.items():
    print(f"{class_name}: {class_weights_framepad[class_id]:.3f}")
# Create a fresh CNN for the frame-padded features
framepad_model = keras.Sequential([
    layers.Input(shape=X_train_framepad.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(len(CLASSES), activation="softmax")
])
# Configure the model for training
framepad_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
# Display the architecture
framepad_model.summary()


In [ ]:
from tensorflow import keras

framepad_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    # Stop if validation loss stops improving
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate if validation loss plateaus
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_framepad = framepad_model.fit(
    X_train_framepad,
    y_train_framepad,
    validation_data=(
        X_val_framepad,
        y_val_framepad
    ),
    epochs=40,
    batch_size=32,
    class_weight=class_weights_framepad,
    callbacks=framepad_callbacks,
    verbose=1
)


### 11.3 Evaluate the weighted model


In [ ]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)
# Predict probabilities for validation segments
val_probabilities = framepad_model.predict(
    X_val_framepad,
    batch_size=32,
    verbose=1
)
# Convert probabilities to predicted class IDs
y_val_pred_framepad = np.argmax(val_probabilities, axis=1)
# Calculate overall metrics
print("Validation accuracy:",
      round(accuracy_score(y_val_framepad, y_val_pred_framepad), 4))

print("Validation macro-F1:",
      round(f1_score(
          y_val_framepad,
          y_val_pred_framepad,
          average="macro",
          zero_division=0
      ), 4))

print("Validation balanced accuracy:",
      round(balanced_accuracy_score(
          y_val_framepad,
          y_val_pred_framepad
      ), 4))
# Detailed performance for each class
print("\nClassification report:")
print(classification_report(
    y_val_framepad,
    y_val_pred_framepad,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    zero_division=0,
    digits=3
))
# Confusion matrix: rows = actual, columns = predicted
cm_framepad = confusion_matrix(
    y_val_framepad,
    y_val_pred_framepad,
    labels=list(range(len(CLASSES)))
)

print("Confusion matrix:")
print("Class order:", CLASSES)
print(cm_framepad)


In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
import numpy as np

# Predict the training examples using the restored best model
train_probabilities_framepad = framepad_model.predict(
    X_train_framepad,
    batch_size=32,
    verbose=1
)

y_train_pred_framepad = np.argmax(
    train_probabilities_framepad,
    axis=1
)

print(
    "Training accuracy:",
    round(accuracy_score(
        y_train_framepad,
        y_train_pred_framepad
    ), 4)
)

print(
    "Training macro-F1:",
    round(f1_score(
        y_train_framepad,
        y_train_pred_framepad,
        average="macro",
        zero_division=0
    ), 4)
)

print("\nTraining classification report:")
print(classification_report(
    y_train_framepad,
    y_train_pred_framepad,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    digits=3,
    zero_division=0
))


### 11.4 Global-average-pooling variant


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Create a fresh model with the same convolution layers
# as the previous frame-padding model.
framepad_gap_model = keras.Sequential([
    layers.Input(shape=X_train_framepad.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    # Change: summarize each feature map instead of flattening it
    layers.GlobalAveragePooling2D(),

    layers.Dense(32, activation="relu"),
    layers.Dense(len(CLASSES), activation="softmax")
])

framepad_gap_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

framepad_gap_model.summary()


In [ ]:
from tensorflow import keras

framepad_gap_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_gap_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_framepad_gap = framepad_gap_model.fit(
    X_train_framepad,
    y_train_framepad,
    validation_data=(
        X_val_framepad,
        y_val_framepad
    ),
    epochs=40,
    batch_size=32,
    class_weight=class_weights_framepad,
    callbacks=framepad_gap_callbacks,
    verbose=1
)


In [ ]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)

# Predict on the same validation set used for the previous model
val_probabilities_gap = framepad_gap_model.predict(
    X_val_framepad,
    batch_size=32,
    verbose=1
)

y_val_pred_gap = np.argmax(val_probabilities_gap, axis=1)

print(
    "Validation accuracy:",
    round(accuracy_score(y_val_framepad, y_val_pred_gap), 4)
)

print(
    "Validation macro-F1:",
    round(f1_score(
        y_val_framepad,
        y_val_pred_gap,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation balanced accuracy:",
    round(balanced_accuracy_score(
        y_val_framepad,
        y_val_pred_gap
    ), 4)
)

print("\nClassification report:")
print(classification_report(
    y_val_framepad,
    y_val_pred_gap,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

print("Confusion matrix (rows = actual, columns = predicted):")
print("Class order:", CLASSES)
print(confusion_matrix(
    y_val_framepad,
    y_val_pred_gap,
    labels=list(range(len(CLASSES)))
))


### 11.5 Dropout variant


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

framepad_dropout_model = keras.Sequential([
    layers.Input(shape=X_train_framepad.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),

    # Regularization: randomly deactivate 30% of activations
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),

    # Additional, lighter regularization
    layers.Dropout(0.2),

    layers.Dense(len(CLASSES), activation="softmax")
])

framepad_dropout_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

framepad_dropout_model.summary()


In [ ]:
from tensorflow import keras

framepad_dropout_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_dropout_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_framepad_dropout = framepad_dropout_model.fit(
    X_train_framepad,
    y_train_framepad,
    validation_data=(
        X_val_framepad,
        y_val_framepad
    ),
    epochs=40,
    batch_size=32,
    class_weight=class_weights_framepad,
    callbacks=framepad_dropout_callbacks,
    verbose=1
)


In [ ]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)

# Predict validation probabilities using the restored best weights
val_probabilities_dropout = framepad_dropout_model.predict(
    X_val_framepad,
    batch_size=32,
    verbose=1
)

# Choose the class with the highest predicted probability
y_val_pred_dropout = np.argmax(
    val_probabilities_dropout,
    axis=1
)

# Overall metrics
print(
    "Validation accuracy:",
    round(accuracy_score(
        y_val_framepad, y_val_pred_dropout
    ), 4)
)

print(
    "Validation macro-F1:",
    round(f1_score(
        y_val_framepad,
        y_val_pred_dropout,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation balanced accuracy:",
    round(balanced_accuracy_score(
        y_val_framepad, y_val_pred_dropout
    ), 4)
)

# Individual class performance
print("\nClassification report:")
print(classification_report(
    y_val_framepad,
    y_val_pred_dropout,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

# Rows = actual labels; columns = predicted labels
print("\nConfusion matrix:")
print("Class order:", CLASSES)
print(confusion_matrix(
    y_val_framepad,
    y_val_pred_dropout,
    labels=list(range(len(CLASSES)))
))


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Predict the training data using the restored best dropout model
train_probabilities_dropout = framepad_dropout_model.predict(
    X_train_framepad,
    batch_size=32,
    verbose=1
)

y_train_pred_dropout = np.argmax(
    train_probabilities_dropout,
    axis=1
)

print(
    "Training accuracy:",
    round(accuracy_score(
        y_train_framepad,
        y_train_pred_dropout
    ), 4)
)

print(
    "Training macro-F1:",
    round(f1_score(
        y_train_framepad,
        y_train_pred_dropout,
        average="macro",
        zero_division=0
    ), 4)
)

print("\nTraining classification report:")
print(classification_report(
    y_train_framepad,
    y_train_pred_dropout,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

print("\nTraining confusion matrix:")
print("Class order:", CLASSES)
print(confusion_matrix(
    y_train_framepad,
    y_train_pred_dropout,
    labels=list(range(len(CLASSES)))
))


### 11.6 Combined three-channel model


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

# Copy the existing weights so we don't change the original
class_weights_combined_boost = class_weights_framepad.copy()

# Increase only the combined-class weight by 50%
combined_id = class_to_idx["crackle+wheeze"]

class_weights_combined_boost[combined_id] *= 1.5

print("Updated class weights:")
for class_id, class_name in idx_to_class.items():
    print(
        f"{class_name}: "
        f"{class_weights_combined_boost[class_id]:.3f}"
    )

# Build a fresh model with the same architecture as before
framepad_combined_boost_model = keras.Sequential([
    layers.Input(shape=X_train_framepad.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(len(CLASSES), activation="softmax")
])

framepad_combined_boost_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

framepad_combined_boost_model.summary()


In [ ]:
from tensorflow import keras

framepad_combined_boost_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_combined_boost_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_framepad_combined_boost = (
    framepad_combined_boost_model.fit(
        X_train_framepad,
        y_train_framepad,
        validation_data=(
            X_val_framepad,
            y_val_framepad
        ),
        epochs=40,
        batch_size=32,
        class_weight=class_weights_combined_boost,
        callbacks=framepad_combined_boost_callbacks,
        verbose=1
    )
)


In [ ]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)

# Predict probabilities on the same validation set
val_probabilities_combined_boost = (
    framepad_combined_boost_model.predict(
        X_val_framepad,
        batch_size=32,
        verbose=1
    )
)

# Convert probabilities to predicted class IDs
y_val_pred_combined_boost = np.argmax(
    val_probabilities_combined_boost,
    axis=1
)

# Overall metrics
print(
    "Validation accuracy:",
    round(accuracy_score(
        y_val_framepad,
        y_val_pred_combined_boost
    ), 4)
)

print(
    "Validation macro-F1:",
    round(f1_score(
        y_val_framepad,
        y_val_pred_combined_boost,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation balanced accuracy:",
    round(balanced_accuracy_score(
        y_val_framepad,
        y_val_pred_combined_boost
    ), 4)
)

# Metrics for each of the four classes
print("\nClassification report:")
print(classification_report(
    y_val_framepad,
    y_val_pred_combined_boost,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

# Rows = actual class; columns = predicted class
print("\nConfusion matrix:")
print("Class order:", CLASSES)
print(confusion_matrix(
    y_val_framepad,
    y_val_pred_combined_boost,
    labels=list(range(len(CLASSES)))
))


### 11.7 Represent four classes as two respiratory events


In [ ]:
import numpy as np

crackle_id = class_to_idx["crackle"]
wheeze_id = class_to_idx["wheeze"]
combined_id = class_to_idx["crackle+wheeze"]


def make_multilabel_targets(class_ids):
    # Is a crackle present in each segment?
    crackle_present = np.isin(
        class_ids,
        [crackle_id, combined_id]
    )

    # Is a wheeze present in each segment?
    wheeze_present = np.isin(
        class_ids,
        [wheeze_id, combined_id]
    )

    # Each row becomes [crackle_present, wheeze_present]
    return np.column_stack(
        [crackle_present, wheeze_present]
    ).astype(np.float32)


# Transform training and validation labels separately
y_train_multi = make_multilabel_targets(y_train_framepad)
y_val_multi = make_multilabel_targets(y_val_framepad)


print("Training multilabel shape:", y_train_multi.shape)
print("Validation multilabel shape:", y_val_multi.shape)

print("\nTraining event counts:")
print("Crackle present:", int(y_train_multi[:, 0].sum()))
print("Wheeze present:", int(y_train_multi[:, 1].sum()))

print("\nValidation event counts:")
print("Crackle present:", int(y_val_multi[:, 0].sum()))
print("Wheeze present:", int(y_val_multi[:, 1].sum()))

print("\nValidation label combinations:")
for pattern, name in [
    ([0, 0], "Normal"),
    ([1, 0], "Crackle only"),
    ([0, 1], "Wheeze only"),
    ([1, 1], "Crackle + wheeze")
]:
    count = np.all(y_val_multi == pattern, axis=1).sum()
    print(f"{name}: {count}")


### 11.8 Train the multi-label CNN


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

# Build a fresh multi-label CNN
framepad_multilabel_model = keras.Sequential([
    layers.Input(shape=X_train_framepad.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    # Two independent probabilities: crackle and wheeze
    layers.Dense(2, activation="sigmoid")
])

framepad_multilabel_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

framepad_multilabel_model.summary()


In [ ]:
from tensorflow import keras

multilabel_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_multilabel_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_multilabel = framepad_multilabel_model.fit(
    X_train_framepad,
    y_train_multi,
    validation_data=(
        X_val_framepad,
        y_val_multi
    ),
    epochs=40,
    batch_size=32,
    callbacks=multilabel_callbacks,
    verbose=1
)


In [ ]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)
# Get the two predicted probabilities for each validation segment
val_probs_multi = framepad_multilabel_model.predict(
    X_val_framepad,
    batch_size=32,
    verbose=1
)
# Apply the standard 0.5 threshold to each event
# Column 0 = crackle; column 1 = wheeze
y_val_pred_multi = (val_probs_multi >= 0.5).astype(int)
# Evaluate the two sound events independently
print("EVENT-LEVEL RESULTS")
print(classification_report(
    y_val_multi.astype(int),
    y_val_pred_multi,
    target_names=["crackle", "wheeze"],
    digits=3,
    zero_division=0
))
# Convert the two binary predictions back into four class IDs
# [0, 0] = normal
# [1, 0] = crackle
# [0, 1] = wheeze
# [1, 1] = crackle+wheeze

def multilabel_to_class_ids(binary_predictions):
    crackle_present = binary_predictions[:, 0] == 1
    wheeze_present = binary_predictions[:, 1] == 1

    predicted_classes = np.zeros(
        len(binary_predictions),
        dtype=int
    )

    predicted_classes[
        crackle_present & ~wheeze_present
    ] = class_to_idx["crackle"]

    predicted_classes[
        ~crackle_present & wheeze_present
    ] = class_to_idx["wheeze"]

    predicted_classes[
        crackle_present & wheeze_present
    ] = class_to_idx["crackle+wheeze"]

    return predicted_classes


y_val_pred_multi_four = multilabel_to_class_ids(
    y_val_pred_multi
)
# Evaluate against the original four-class labels
print("\nFOUR-CLASS RESULTS")

print(
    "Validation accuracy:",
    round(accuracy_score(
        y_val_framepad,
        y_val_pred_multi_four
    ), 4)
)

print(
    "Validation macro-F1:",
    round(f1_score(
        y_val_framepad,
        y_val_pred_multi_four,
        average="macro",
        zero_division=0
    ), 4)
)

print(
    "Validation balanced accuracy:",
    round(balanced_accuracy_score(
        y_val_framepad,
        y_val_pred_multi_four
    ), 4)
)

print("\nClassification report:")
print(classification_report(
    y_val_framepad,
    y_val_pred_multi_four,
    labels=list(range(len(CLASSES))),
    target_names=CLASSES,
    digits=3,
    zero_division=0
))

print("\nConfusion matrix (rows = actual, columns = predicted):")
print("Class order:", CLASSES)
print(confusion_matrix(
    y_val_framepad,
    y_val_pred_multi_four,
    labels=list(range(len(CLASSES)))
))

print("\nPredicted four-class counts:")
for class_id, class_name in idx_to_class.items():
    count = np.sum(y_val_pred_multi_four == class_id)
    print(f"{class_name}: {count}")


### 11.9 Sweep event thresholds on validation data


In [ ]:
import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

event_names = ["crackle", "wheeze"]
thresholds = np.arange(0.05, 0.96, 0.05)

for event_id, event_name in enumerate(event_names):

    # Actual presence/absence of this event
    y_true = y_val_multi[:, event_id].astype(int)

    # Model's predicted probability for this event
    probabilities = val_probs_multi[:, event_id]

    print(f"\n{'=' * 45}")
    print(f"EVENT: {event_name.upper()}")
    print(f"{'=' * 45}")

    # Compare scores for examples with and without the event
    print(
        "Mean probability when absent:",
        round(float(probabilities[y_true == 0].mean()), 4)
    )
    print(
        "Mean probability when present:",
        round(float(probabilities[y_true == 1].mean()), 4)
    )
    print(
        "Minimum / maximum probability:",
        round(float(probabilities.min()), 4),
        "/",
        round(float(probabilities.max()), 4)
    )

    # Explore different decision thresholds
    results = []

    for threshold in thresholds:
        y_pred = (probabilities >= threshold).astype(int)

        results.append({
            "threshold": threshold,
            "precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "f1": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "predicted_positive": int(y_pred.sum())
        })

    print("\nThreshold comparison:")
    print("threshold | precision | recall | F1 | predicted positive")

    for result in results:
        print(
            f"{result['threshold']:.2f}      | "
            f"{result['precision']:.3f}     | "
            f"{result['recall']:.3f}  | "
            f"{result['f1']:.3f} | "
            f"{result['predicted_positive']}"
        )

    best_result = max(results, key=lambda row: row["f1"])

    print("\nBest threshold on this validation sweep:")
    print(best_result)


## 12. Event-level metrics and threshold evaluation

### 12.1 Evaluate event scores without fixing a threshold


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

event_names = ["crackle", "wheeze"]

print("Threshold-independent evaluation")
print("-" * 45)

for event_id, event_name in enumerate(event_names):
    y_true = y_val_multi[:, event_id].astype(int)
    probabilities = val_probs_multi[:, event_id]

    roc_auc = roc_auc_score(y_true, probabilities)
    average_precision = average_precision_score(
        y_true, probabilities
    )

    positive_rate = y_true.mean()

    print(f"\n{event_name.upper()}")
    print(f"Positive segments: {y_true.sum()} / {len(y_true)}")
    print(f"Positive prevalence: {positive_rate:.3f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"Average precision: {average_precision:.4f}")


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Get predictions on the training data using the restored best model
train_probs_multi = framepad_multilabel_model.predict(
    X_train_framepad,
    batch_size=32,
    verbose=1
)

event_names = ["crackle", "wheeze"]

print("Training vs validation comparison")
print("-" * 55)

for event_id, event_name in enumerate(event_names):
    train_true = y_train_multi[:, event_id].astype(int)
    val_true = y_val_multi[:, event_id].astype(int)

    train_auc = roc_auc_score(
        train_true, train_probs_multi[:, event_id]
    )
    val_auc = roc_auc_score(
        val_true, val_probs_multi[:, event_id]
    )

    train_ap = average_precision_score(
        train_true, train_probs_multi[:, event_id]
    )
    val_ap = average_precision_score(
        val_true, val_probs_multi[:, event_id]
    )

    print(f"\n{event_name.upper()}")
    print(f"Training ROC-AUC:   {train_auc:.4f}")
    print(f"Validation ROC-AUC: {val_auc:.4f}")
    print(f"Training AP:        {train_ap:.4f}")
    print(f"Validation AP:      {val_ap:.4f}")


### 12.2 Global-pooling multi-label variant


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

framepad_multilabel_gap_model = keras.Sequential([
    layers.Input(shape=X_train_framepad.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    # Reduce each feature map to one value
    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),
    layers.Dense(24, activation="relu"),
    layers.Dropout(0.3),

    # Independent probabilities for crackle and wheeze
    layers.Dense(2, activation="sigmoid")
])

framepad_multilabel_gap_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

framepad_multilabel_gap_model.summary()


In [ ]:
from tensorflow import keras

multilabel_gap_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_multilabel_gap.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_multilabel_gap = framepad_multilabel_gap_model.fit(
    X_train_framepad,
    y_train_multi,
    validation_data=(
        X_val_framepad,
        y_val_multi
    ),
    epochs=40,
    batch_size=32,
    callbacks=multilabel_gap_callbacks,
    verbose=1
)


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import precision_score, recall_score, f1_score

# Get the predicted probabilities for the validation data.
# Column 0 = crackle probability; column 1 = wheeze probability.
val_prob_gap = framepad_multilabel_gap_model.predict(
    X_val_framepad,
    batch_size=32,
    verbose=0
)

event_names = ["Crackle", "Wheeze"]

for i, event_name in enumerate(event_names):
    true_labels = y_val_multi[:, i]
    probabilities = val_prob_gap[:, i]

    # Convert probabilities into positive/negative predictions.
    predicted_labels = (probabilities >= 0.5).astype(int)

    print(f"\n--- {event_name} ---")
    print(f"ROC-AUC: {roc_auc_score(true_labels, probabilities):.3f}")
    print(f"Average precision: {average_precision_score(true_labels, probabilities):.3f}")
    print(f"Actual positive cases: {int(true_labels.sum())}")
    print(f"Predicted positive cases: {int(predicted_labels.sum())}")
    print(f"Mean probability when absent: {probabilities[true_labels == 0].mean():.3f}")
    print(f"Mean probability when present: {probabilities[true_labels == 1].mean():.3f}")
    print(f"Precision at 0.5 threshold: {precision_score(true_labels, predicted_labels, zero_division=0):.3f}")
    print(f"Recall at 0.5 threshold: {recall_score(true_labels, predicted_labels, zero_division=0):.3f}")
    print(f"F1-score at 0.5 threshold: {f1_score(true_labels, predicted_labels, zero_division=0):.3f}")


### 12.3 MFCC feature comparison


In [ ]:
import numpy as np
import librosa
import soundfile as sf


def build_mfcc_features_framepad(frame, n_mfcc=40, target_frames=157):
    features = []
    labels = []
    frame_masks = []
    audio_cache = {}

    for row in frame.itertuples(index=False):
        wav_path = str(row.wav_path)

        # Load and resample each audio recording only once.
        if wav_path not in audio_cache:
            audio, original_sr = sf.read(
                wav_path, dtype="float32"
            )

            # Convert stereo audio to mono if necessary.
            if audio.ndim > 1:
                audio = audio.mean(axis=1)

            # Convert audio to our common sampling rate.
            if original_sr != TARGET_SR:
                audio = librosa.resample(
                    audio,
                    orig_sr=original_sr,
                    target_sr=TARGET_SR
                )

            audio_cache[wav_path] = audio

        audio = audio_cache[wav_path]

        # Convert annotation times from seconds to sample indices.
        start_sample = max(
            0, int(round(float(row.start) * TARGET_SR))
        )
        end_sample = min(
            len(audio), int(round(float(row.end) * TARGET_SR))
        )

        segment = audio[start_sample:end_sample]

        if len(segment) == 0:
            raise ValueError(
                f"Empty audio segment: {row.recording}"
            )

        # Crop only if the segment exceeds five seconds.
        if len(segment) > TARGET_SAMPLES:
            segment = fit_audio_length(
                segment, TARGET_SAMPLES
            )

        # Generate the same Mel power representation used earlier.
        mel_power = librosa.feature.melspectrogram(
            y=segment,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0
        )

        # Convert power to decibels, then calculate MFCCs.
        log_mel_db = librosa.power_to_db(
            mel_power, ref=np.max
        )

        mfcc = librosa.feature.mfcc(
            S=log_mel_db,
            n_mfcc=n_mfcc
        )

        # Track which time frames contain actual features.
        # This lets us normalise features without treating
        # padding as real audio.
        actual_frames = mfcc.shape[1]
        mask = np.zeros(target_frames, dtype=bool)

        if actual_frames < target_frames:
            pad_left = (target_frames - actual_frames) // 2
            mask[pad_left:pad_left + actual_frames] = True
        else:
            mask[:] = True

        # Match the time dimension used by our log-Mel arrays.
        mfcc = fit_logmel_frames(mfcc, target_frames)

        # CNN input requires a final channel dimension.
        features.append(mfcc[..., np.newaxis].astype(np.float32))
        frame_masks.append(mask)
        labels.append(class_to_idx[row.label])

    return (
        np.stack(features),
        np.asarray(labels, dtype=np.int64),
        np.stack(frame_masks)
    )


# Extract training and validation features separately.
X_train_mfcc_raw, y_train_mfcc, train_mfcc_mask = (
    build_mfcc_features_framepad(candidate_train_df)
)

X_val_mfcc_raw, y_val_mfcc, val_mfcc_mask = (
    build_mfcc_features_framepad(candidate_val_df)
)

# Confirm that labels and row ordering match our log-Mel setup.
assert np.array_equal(y_train_mfcc, y_train_framepad)
assert np.array_equal(y_val_mfcc, y_val_framepad)

print("Training MFCC shape:", X_train_mfcc_raw.shape)
print("Validation MFCC shape:", X_val_mfcc_raw.shape)

print("Training labels match log-Mel:", True)
print("Validation labels match log-Mel:", True)

print(
    "Training actual-frame percentage:",
    round(100 * train_mfcc_mask.mean(), 2), "%"
)
print(
    "Validation actual-frame percentage:",
    round(100 * val_mfcc_mask.mean(), 2), "%"
)

print("Training features finite:",
      np.isfinite(X_train_mfcc_raw).all())
print("Validation features finite:",
      np.isfinite(X_val_mfcc_raw).all())


In [ ]:
# Access the MFCC arrays without creating additional copies.
X_train_mfcc = X_train_mfcc_raw
X_val_mfcc = X_val_mfcc_raw

# Remove the last channel dimension temporarily.
# Shape: (segments, MFCC coefficients, time frames)
train_values = X_train_mfcc[..., 0]

# Calculate mean and standard deviation for each coefficient,
# using only genuine feature frames from the training set.
n_coefficients = train_values.shape[1]

mfcc_mean = np.array([
    train_values[:, i, :][train_mfcc_mask].mean()
    for i in range(n_coefficients)
])

mfcc_std = np.array([
    train_values[:, i, :][train_mfcc_mask].std()
    for i in range(n_coefficients)
])

# Protect against division by zero.
mfcc_std_safe = np.maximum(mfcc_std, 1e-6)

# Normalize training features in place to save memory.
X_train_mfcc[..., 0] -= mfcc_mean[None, :, None]
X_train_mfcc[..., 0] /= mfcc_std_safe[None, :, None]

# Restore padded time frames to exactly zero.
X_train_mfcc[..., 0] *= train_mfcc_mask[:, None, :]

# Apply the SAME training statistics to validation.
X_val_mfcc[..., 0] -= mfcc_mean[None, :, None]
X_val_mfcc[..., 0] /= mfcc_std_safe[None, :, None]

# Keep validation padding at zero as well.
X_val_mfcc[..., 0] *= val_mfcc_mask[:, None, :]

print("Training shape:", X_train_mfcc.shape)
print("Validation shape:", X_val_mfcc.shape)

print("\nTraining features finite:",
      np.isfinite(X_train_mfcc).all())
print("Validation features finite:",
      np.isfinite(X_val_mfcc).all())

normalised_means = np.array([
    X_train_mfcc[:, i, :, 0][train_mfcc_mask].mean()
    for i in range(n_coefficients)
])

normalised_stds = np.array([
    X_train_mfcc[:, i, :, 0][train_mfcc_mask].std()
    for i in range(n_coefficients)
])

print("\nMean across coefficients:",
      round(normalised_means.mean(), 4))
print("Average standard deviation across coefficients:",
      round(normalised_stds.mean(), 4))

print("\nAll training labels still match:",
      np.array_equal(y_train_mfcc, y_train_framepad))
print("All validation labels still match:",
      np.array_equal(y_val_mfcc, y_val_framepad))


### 12.4 Train the MFCC CNN


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Make the model initialization reproducible.
keras.utils.set_random_seed(42)

# Build a CNN that predicts crackle and wheeze independently.
framepad_multilabel_mfcc_model = keras.Sequential([
    layers.Input(shape=X_train_mfcc.shape[1:]),

    # Learn local time-frequency patterns.
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    # Convert learned feature maps into a feature vector.
    layers.Flatten(),

    # Reduce overfitting.
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    # Output probabilities: crackle and wheeze.
    layers.Dense(2, activation="sigmoid")
])

framepad_multilabel_mfcc_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

framepad_multilabel_mfcc_model.summary()


In [ ]:
from tensorflow import keras

mfcc_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_multilabel_mfcc.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_multilabel_mfcc = framepad_multilabel_mfcc_model.fit(
    X_train_mfcc,
    y_train_multi,
    validation_data=(X_val_mfcc, y_val_multi),
    epochs=40,
    batch_size=32,
    callbacks=mfcc_callbacks,
    verbose=1
)


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Predict using the best weights restored by early stopping.
val_prob_mfcc = framepad_multilabel_mfcc_model.predict(
    X_val_mfcc,
    batch_size=32,
    verbose=0
)

event_names = ["Crackle", "Wheeze"]

for i, event_name in enumerate(event_names):
    true_labels = y_val_multi[:, i]
    probabilities = val_prob_mfcc[:, i]

    predicted_labels = (probabilities >= 0.5).astype(int)

    print(f"\n--- {event_name} (MFCC) ---")
    print(f"ROC-AUC: {roc_auc_score(true_labels, probabilities):.3f}")
    print(
        f"Average precision: "
        f"{average_precision_score(true_labels, probabilities):.3f}"
    )
    print(f"Actual positive cases: {int(true_labels.sum())}")
    print(f"Predicted positive cases: {int(predicted_labels.sum())}")
    print(
        f"Mean probability when absent: "
        f"{probabilities[true_labels == 0].mean():.3f}"
    )
    print(
        f"Mean probability when present: "
        f"{probabilities[true_labels == 1].mean():.3f}"
    )
    print(
        f"Precision at 0.5: "
        f"{precision_score(true_labels, predicted_labels, zero_division=0):.3f}"
    )
    print(
        f"Recall at 0.5: "
        f"{recall_score(true_labels, predicted_labels, zero_division=0):.3f}"
    )
    print(
        f"F1-score at 0.5: "
        f"{f1_score(true_labels, predicted_labels, zero_division=0):.3f}"
    )


### 12.5 Log-Mel and MFCC fusion


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

# Separate inputs for the two feature representations.
mel_input = keras.Input(
    shape=X_train_framepad.shape[1:],
    name="logmel_input"
)

mfcc_input = keras.Input(
    shape=X_train_mfcc.shape[1:],
    name="mfcc_input"
)


# Branch 1: learn patterns from log-Mel features.
mel_x = layers.Conv2D(
    16, (3, 3), activation="relu", padding="same"
)(mel_input)
mel_x = layers.MaxPooling2D((2, 2))(mel_x)

mel_x = layers.Conv2D(
    32, (3, 3), activation="relu", padding="same"
)(mel_x)
mel_x = layers.MaxPooling2D((2, 2))(mel_x)

mel_x = layers.Conv2D(
    64, (3, 3), activation="relu", padding="same"
)(mel_x)
mel_x = layers.AveragePooling2D((4, 4))(mel_x)
mel_x = layers.Flatten()(mel_x)


# Branch 2: learn patterns from MFCC features.
mfcc_x = layers.Conv2D(
    16, (3, 3), activation="relu", padding="same"
)(mfcc_input)
mfcc_x = layers.MaxPooling2D((2, 2))(mfcc_x)

mfcc_x = layers.Conv2D(
    32, (3, 3), activation="relu", padding="same"
)(mfcc_x)
mfcc_x = layers.MaxPooling2D((2, 2))(mfcc_x)

mfcc_x = layers.Conv2D(
    64, (3, 3), activation="relu", padding="same"
)(mfcc_x)
mfcc_x = layers.AveragePooling2D((4, 4))(mfcc_x)
mfcc_x = layers.Flatten()(mfcc_x)


# Combine the learned representations.
combined = layers.Concatenate(name="feature_fusion")(
    [mel_x, mfcc_x]
)

combined = layers.Dropout(0.3)(combined)
combined = layers.Dense(32, activation="relu")(combined)
combined = layers.Dropout(0.2)(combined)

# Independent probabilities for crackles and wheezes.
output = layers.Dense(2, activation="sigmoid")(combined)

framepad_fusion_model = keras.Model(
    inputs=[mel_input, mfcc_input],
    outputs=output,
    name="logmel_mfcc_fusion"
)

framepad_fusion_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

framepad_fusion_model.summary()


In [ ]:
from tensorflow import keras

fusion_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_framepad_fusion.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history_fusion = framepad_fusion_model.fit(
    [X_train_framepad, X_train_mfcc],
    y_train_multi,
    validation_data=(
        [X_val_framepad, X_val_mfcc],
        y_val_multi
    ),
    epochs=40,
    batch_size=32,
    callbacks=fusion_callbacks,
    verbose=1
)


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Predict using the best weights restored by early stopping.
val_prob_fusion = framepad_fusion_model.predict(
    [X_val_framepad, X_val_mfcc],
    batch_size=32,
    verbose=0
)

event_names = ["Crackle", "Wheeze"]

for i, event_name in enumerate(event_names):
    true_labels = y_val_multi[:, i]
    probabilities = val_prob_fusion[:, i]

    predicted_labels = (probabilities >= 0.5).astype(int)

    print(f"\n--- {event_name} (Fusion) ---")
    print(
        f"ROC-AUC: "
        f"{roc_auc_score(true_labels, probabilities):.3f}"
    )
    print(
        f"Average precision: "
        f"{average_precision_score(true_labels, probabilities):.3f}"
    )
    print(f"Actual positive cases: {int(true_labels.sum())}")
    print(f"Predicted positive cases: {int(predicted_labels.sum())}")
    print(
        f"Mean probability when absent: "
        f"{probabilities[true_labels == 0].mean():.3f}"
    )
    print(
        f"Mean probability when present: "
        f"{probabilities[true_labels == 1].mean():.3f}"
    )
    print(
        f"Precision at 0.5: "
        f"{precision_score(true_labels, predicted_labels, zero_division=0):.3f}"
    )
    print(
        f"Recall at 0.5: "
        f"{recall_score(true_labels, predicted_labels, zero_division=0):.3f}"
    )
    print(
        f"F1-score at 0.5: "
        f"{f1_score(true_labels, predicted_labels, zero_division=0):.3f}"
    )


### 12.6 Convert event predictions back to four classes


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

# Convert the two event probabilities into four classes.
predicted_events = (val_prob_fusion >= 0.5).astype(int)

fusion_pred_4class = np.full(
    len(predicted_events),
    class_to_idx["normal"],
    dtype=np.int64
)

fusion_pred_4class[
    (predicted_events[:, 0] == 1) &
    (predicted_events[:, 1] == 0)
] = class_to_idx["crackle"]

fusion_pred_4class[
    (predicted_events[:, 0] == 0) &
    (predicted_events[:, 1] == 1)
] = class_to_idx["wheeze"]

fusion_pred_4class[
    (predicted_events[:, 0] == 1) &
    (predicted_events[:, 1] == 1)
] = class_to_idx["crackle+wheeze"]

actual = y_val_framepad

labels_order = [
    class_to_idx["normal"],
    class_to_idx["crackle"],
    class_to_idx["wheeze"],
    class_to_idx["crackle+wheeze"]
]

cm = confusion_matrix(
    actual,
    fusion_pred_4class,
    labels=labels_order
)

# Specificity for the normal class.
specificity = cm[0, 0] / cm[0, :].sum()

# Pooled sensitivity over all abnormal classes.
abnormal_mask = actual != class_to_idx["normal"]

sensitivity = np.sum(
    (actual == fusion_pred_4class) & abnormal_mask
) / abnormal_mask.sum()

icbhi_score = (specificity + sensitivity) / 2

print("Validation accuracy:", round(accuracy_score(actual, fusion_pred_4class), 4))
print("Macro F1:", round(f1_score(actual, fusion_pred_4class, average="macro"), 4))
print("Abnormal sensitivity:", round(sensitivity, 4))
print("Normal specificity:", round(specificity, 4))
print("ICBHI score:", round(icbhi_score, 4))

print("\nConfusion matrix:")
print(cm)


### 12.7 Freeze validation-selected thresholds


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

normal_id = class_to_idx["normal"]
crackle_id = class_to_idx["crackle"]
wheeze_id = class_to_idx["wheeze"]
both_id = class_to_idx["crackle+wheeze"]

labels_order = [
    normal_id, crackle_id, wheeze_id, both_id
]

actual = y_val_framepad

# Threshold combinations to investigate.
thresholds = [
    0.15, 0.20, 0.25, 0.30,
    0.35, 0.40, 0.45, 0.50
]

results = []

for crackle_threshold in thresholds:
    for wheeze_threshold in thresholds:

        # Predict each respiratory event independently.
        predicted_crackle = (
            val_prob_fusion[:, 0] >= crackle_threshold
        )

        predicted_wheeze = (
            val_prob_fusion[:, 1] >= wheeze_threshold
        )

        # Convert the event predictions into four classes.
        predicted = np.full(
            len(actual),
            normal_id,
            dtype=np.int64
        )

        predicted[
            predicted_crackle & ~predicted_wheeze
        ] = crackle_id

        predicted[
            ~predicted_crackle & predicted_wheeze
        ] = wheeze_id

        predicted[
            predicted_crackle & predicted_wheeze
        ] = both_id

        cm = confusion_matrix(
            actual,
            predicted,
            labels=labels_order
        )

        # Official ICBHI sensitivity and specificity.
        specificity = cm[0, 0] / cm[0, :].sum()

        sensitivity = (
            cm[1, 1] + cm[2, 2] + cm[3, 3]
        ) / cm[1:, :].sum()

        icbhi_score = (
            sensitivity + specificity
        ) / 2

        macro_f1 = f1_score(
            actual,
            predicted,
            labels=labels_order,
            average="macro",
            zero_division=0
        )

        results.append({
            "Crackle threshold": crackle_threshold,
            "Wheeze threshold": wheeze_threshold,
            "ICBHI score": icbhi_score,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Macro F1": macro_f1,
            "Predicted normal": int(np.sum(predicted == normal_id)),
            "Predicted crackle": int(np.sum(predicted == crackle_id)),
            "Predicted wheeze": int(np.sum(predicted == wheeze_id)),
            "Predicted both": int(np.sum(predicted == both_id))
        })

threshold_scan_df = pd.DataFrame(results)

display(
    threshold_scan_df
    .sort_values(
        ["ICBHI score", "Macro F1"],
        ascending=False
    )
    .head(15)
    .round(3)
    .reset_index(drop=True)
)


### 12.8 Build the final three-channel representation


In [ ]:
import numpy as np
import librosa

def build_three_channel_features(
    X_logmel,
    frame_masks,
    split_name
):
    n_samples, n_mels, n_frames, _ = X_logmel.shape

    # Channel 1: original log-Mel
    # Channel 2: first-order delta
    # Channel 3: second-order delta-delta
    X_3ch = np.zeros(
        (n_samples, n_mels, n_frames, 3),
        dtype=np.float32
    )

    X_3ch[..., 0] = X_logmel[..., 0]

    for i in range(n_samples):

        # Identify genuine (non-padded) time frames.
        mask = frame_masks[i]
        valid_mel = X_logmel[i, :, :, 0][:, mask]

        n_valid_frames = valid_mel.shape[1]

        # Librosa requires an odd delta window of at least 3.
        width = min(
            9,
            n_valid_frames if n_valid_frames % 2 == 1
            else n_valid_frames - 1
        )

        if width < 3:
            raise ValueError(
                f"Too few valid frames in sample {i}: "
                f"{n_valid_frames}"
            )

        # First derivative: how spectral energy changes.
        delta = librosa.feature.delta(
            valid_mel,
            width=width,
            order=1,
            axis=-1
        )

        # Second derivative: how the rate of change changes.
        delta_delta = librosa.feature.delta(
            valid_mel,
            width=width,
            order=2,
            axis=-1
        )

        # Put derivatives back in their original time positions.
        X_3ch[i, :, :, 1][:, mask] = delta
        X_3ch[i, :, :, 2][:, mask] = delta_delta

        if (i + 1) % 500 == 0:
            print(f"{split_name}: {i + 1}/{n_samples}")

    return X_3ch


# Build training and validation features separately.
X_train_3ch = build_three_channel_features(
    X_train_framepad,
    train_mfcc_mask,
    "Training"
)

X_val_3ch = build_three_channel_features(
    X_val_framepad,
    val_mfcc_mask,
    "Validation"
)

print("\nTraining shape:", X_train_3ch.shape)
print("Validation shape:", X_val_3ch.shape)

print(
    "Training features finite:",
    np.isfinite(X_train_3ch).all()
)

print(
    "Validation features finite:",
    np.isfinite(X_val_3ch).all()
)


## 13. CNN embeddings with classical classifiers

### 13.1 Prepare normalized three-channel inputs


In [ ]:
# Store normalization statistics for the two derivative channels.
delta_means = {}
delta_stds = {}

for channel in [1, 2]:

    # Calculate one mean and standard deviation per frequency bin.
    channel_means = np.array([
        X_train_3ch[:, f, :, channel][train_mfcc_mask].mean()
        for f in range(X_train_3ch.shape[1])
    ])

    channel_stds = np.array([
        X_train_3ch[:, f, :, channel][train_mfcc_mask].std()
        for f in range(X_train_3ch.shape[1])
    ])

    # Prevent division by zero.
    channel_stds = np.maximum(channel_stds, 1e-6)

    delta_means[channel] = channel_means
    delta_stds[channel] = channel_stds

    # Normalize training features in place.
    X_train_3ch[..., channel] -= channel_means[None, :, None]
    X_train_3ch[..., channel] /= channel_stds[None, :, None]

    # Restore padded frames to zero.
    X_train_3ch[..., channel] *= train_mfcc_mask[:, None, :]

    # Use the SAME training statistics for validation.
    X_val_3ch[..., channel] -= channel_means[None, :, None]
    X_val_3ch[..., channel] /= channel_stds[None, :, None]

    # Preserve zero-padding in validation.
    X_val_3ch[..., channel] *= val_mfcc_mask[:, None, :]


print("Training shape:", X_train_3ch.shape)
print("Validation shape:", X_val_3ch.shape)

for channel, name in [
    (0, "Log-Mel"),
    (1, "Delta"),
    (2, "Delta-delta")
]:
    actual_values = np.concatenate([
        X_train_3ch[:, f, :, channel][train_mfcc_mask]
        for f in range(X_train_3ch.shape[1])
    ])

    print(
        f"{name}: mean={actual_values.mean():.4f}, "
        f"std={actual_values.std():.4f}"
    )

print("\nTraining finite:",
      np.isfinite(X_train_3ch).all())

print("Validation finite:",
      np.isfinite(X_val_3ch).all())

print(
    "Original Log-Mel channel preserved:",
    np.allclose(X_train_3ch[..., 0], X_train_framepad[..., 0])
)

print(
    "Training labels match:",
    np.array_equal(y_train_framepad, y_train_multi[:, 0] * 0 + y_train_framepad)
)


### 13.2 Train the patient-balanced three-channel CNN


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

expected_y_train_multi = np.column_stack([
    np.isin(
        y_train_framepad,
        [class_to_idx["crackle"], class_to_idx["crackle+wheeze"]]
    ),
    np.isin(
        y_train_framepad,
        [class_to_idx["wheeze"], class_to_idx["crackle+wheeze"]]
    )
]).astype(np.float32)

expected_y_val_multi = np.column_stack([
    np.isin(
        y_val_framepad,
        [class_to_idx["crackle"], class_to_idx["crackle+wheeze"]]
    ),
    np.isin(
        y_val_framepad,
        [class_to_idx["wheeze"], class_to_idx["crackle+wheeze"]]
    )
]).astype(np.float32)

assert np.array_equal(y_train_multi, expected_y_train_multi)
assert np.array_equal(y_val_multi, expected_y_val_multi)

assert X_train_3ch.shape[0] == len(y_train_multi)
assert X_val_3ch.shape[0] == len(y_val_multi)

print("Training labels correctly aligned:", True)
print("Validation labels correctly aligned:", True)


# Set a reproducible initialization.
keras.utils.set_random_seed(42)

# Create the three-channel CNN.
framepad_multilabel_3ch_model = keras.Sequential([
    layers.Input(shape=X_train_3ch.shape[1:]),

    layers.Conv2D(
        16, (3, 3), activation="relu", padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        32, (3, 3), activation="relu", padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        64, (3, 3), activation="relu", padding="same"
    ),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),

    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(2, activation="sigmoid")
])

framepad_multilabel_3ch_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

framepad_multilabel_3ch_model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks_3ch = [
    EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]

history_3ch = framepad_multilabel_3ch_model.fit(
    X_train_3ch,
    y_train_multi,

    validation_data=(
        X_val_3ch,
        y_val_multi
    ),

    epochs=50,
    batch_size=32,
    callbacks=callbacks_3ch,
    shuffle=True,
    verbose=1
)

# Save the model with the best validation weights restored by EarlyStopping.
framepad_multilabel_3ch_model.save(
    "best_3ch_multilabel.keras"
)

print("\nTraining completed.")
print("Best validation loss:",
      min(history_3ch.history["val_loss"]))

print("Best validation binary accuracy:",
      max(history_3ch.history["val_binary_accuracy"]))


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Calculate positive and negative counts for each event.
positive_counts = np.sum(y_train_multi, axis=0)
negative_counts = len(y_train_multi) - positive_counts

# Give more importance to positive examples.
positive_weights = negative_counts / positive_counts

print("Crackle positive weight:", round(positive_weights[0], 3))
print("Wheeze positive weight:", round(positive_weights[1], 3))

pos_weights_tf = tf.constant(
    positive_weights,
    dtype=tf.float32
)


# Custom weighted binary cross-entropy.
def weighted_binary_crossentropy(y_true, y_pred):

    y_true = tf.cast(y_true, tf.float32)

    epsilon = keras.backend.epsilon()

    y_pred = tf.clip_by_value(
        y_pred,
        epsilon,
        1.0 - epsilon
    )

    loss = -(
        y_true * pos_weights_tf * tf.math.log(y_pred)
        +
        (1.0 - y_true) * tf.math.log(1.0 - y_pred)
    )

    return tf.reduce_mean(loss, axis=-1)


# Build the same three-channel CNN architecture.
keras.utils.set_random_seed(42)

weighted_3ch_model = keras.Sequential([
    layers.Input(shape=X_train_3ch.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),

    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(2, activation="sigmoid")
])

weighted_3ch_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)


# Early stopping and learning-rate reduction.
callbacks_weighted = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]


# Train.
history_weighted_3ch = weighted_3ch_model.fit(
    X_train_3ch,
    y_train_multi,
    validation_data=(X_val_3ch, y_val_multi),
    epochs=50,
    batch_size=32,
    callbacks=callbacks_weighted,
    shuffle=True,
    verbose=1
)

print("\nBest validation loss:",
      min(history_weighted_3ch.history["val_loss"]))

print("Best validation binary accuracy:",
      max(history_weighted_3ch.history["val_binary_accuracy"]))


In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers

# --------------------------------------------------
# Reconstruct the exact metadata order used by
#    the feature builder (groupby wav_path, sort=False).
# --------------------------------------------------

feature_order_df = pd.concat(
    [
        group
        for _, group in candidate_train_df.groupby(
            "wav_path", sort=False
        )
    ],
    ignore_index=True
)

assert len(feature_order_df) == len(X_train_3ch)

assert np.array_equal(
    feature_order_df["label"].map(class_to_idx).to_numpy(),
    y_train_framepad
)

assert np.array_equal(
    y_train_framepad,
    np.argmax(
        np.column_stack([
            (y_train_multi[:, 0] == 0) &
            (y_train_multi[:, 1] == 0),
            (y_train_multi[:, 0] == 1) &
            (y_train_multi[:, 1] == 0),
            (y_train_multi[:, 0] == 0) &
            (y_train_multi[:, 1] == 1),
            (y_train_multi[:, 0] == 1) &
            (y_train_multi[:, 1] == 1)
        ]),
        axis=1
    )
)

print("Feature and patient metadata alignment: verified")


# --------------------------------------------------
# Recalculate patient weights in feature order.
# --------------------------------------------------

patient_segment_counts = (
    feature_order_df.groupby("patient").size()
)

patient_weights_aligned = (
    feature_order_df["patient"]
    .map(
        lambda patient:
        1.0 / np.sqrt(patient_segment_counts.loc[patient])
    )
    .to_numpy(dtype=np.float32)
)

patient_weights_aligned /= patient_weights_aligned.mean()

assert len(patient_weights_aligned) == len(X_train_3ch)
assert np.isfinite(patient_weights_aligned).all()

print("Average patient weight:",
      round(patient_weights_aligned.mean(), 4))


# --------------------------------------------------
# Create a fresh three-channel CNN.
# --------------------------------------------------

keras.utils.set_random_seed(42)

patient_balanced_3ch_model = keras.Sequential([
    layers.Input(shape=(64, 157, 3)),

    layers.Conv2D(
        16, (3, 3), activation="relu", padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        32, (3, 3), activation="relu", padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        64, (3, 3), activation="relu", padding="same"
    ),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),

    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(2, activation="sigmoid")
])

# Reuse the positive-class-weighted loss from the
# previous experiment. Patient weights are applied
# separately through sample_weight during fitting.
patient_balanced_3ch_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

print("\nModel parameters:",
      patient_balanced_3ch_model.count_params())


# --------------------------------------------------
# Prepare early stopping.
# --------------------------------------------------

callbacks_patient_balanced = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]


# --------------------------------------------------
# Train with patient-level sample weights.
# --------------------------------------------------

history_patient_balanced = (
    patient_balanced_3ch_model.fit(
        X_train_3ch,
        y_train_multi,

        sample_weight=patient_weights_aligned,

        validation_data=(
            X_val_3ch,
            y_val_multi
        ),

        epochs=50,
        batch_size=32,
        callbacks=callbacks_patient_balanced,
        shuffle=True,
        verbose=1
    )
)

patient_balanced_3ch_model.save(
    "best_3ch_patient_balanced.keras"
)

print("\nTraining completed.")
print(
    "Best validation loss:",
    min(history_patient_balanced.history["val_loss"])
)


### 13.3 Extract CNN embeddings


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

models_to_compare = {
    "Weighted model": weighted_3ch_model,
    "Patient-balanced model": patient_balanced_3ch_model
}

class_names = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

for _, row in comparison_df.iterrows():

    model_name = row["Model"]
    model = models_to_compare[model_name]

    crackle_threshold = float(row["Crackle threshold"])
    wheeze_threshold = float(row["Wheeze threshold"])

    probabilities = model.predict(X_val_3ch, verbose=0)

    pred_crackle = probabilities[:, 0] >= crackle_threshold
    pred_wheeze = probabilities[:, 1] >= wheeze_threshold

    predictions = np.select(
        [
            (~pred_crackle) & (~pred_wheeze),
            pred_crackle & (~pred_wheeze),
            (~pred_crackle) & pred_wheeze,
            pred_crackle & pred_wheeze
        ],
        [0, 1, 2, 3],
        default=0
    )

    print("\n" + "=" * 60)
    print(model_name)
    print("Crackle threshold:", crackle_threshold)
    print("Wheeze threshold:", wheeze_threshold)
    print("=" * 60)

    print("\nConfusion matrix:")
    print(confusion_matrix(
        y_val_framepad,
        predictions,
        labels=[0, 1, 2, 3]
    ))

    print("\nClass-wise report:")
    print(classification_report(
        y_val_framepad,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    ))


### 13.4 Prepare an explicit feature-extractor model


In [ ]:
# Extract the output of the 32-unit Dense layer (Layer 8)

feature_extractor = tf.keras.Model(
    inputs=patient_balanced_3ch_model.input,
    outputs=patient_balanced_3ch_model.layers[8].output
)

# Extract training features
X_train_embeddings = feature_extractor.predict(
    X_train_3ch,
    batch_size=32,
    verbose=1
)

# Extract validation features
X_val_embeddings = feature_extractor.predict(
    X_val_3ch,
    batch_size=32,
    verbose=1
)

print("\nFeature extraction completed.")

print("Training embeddings:", X_train_embeddings.shape)
print("Validation embeddings:", X_val_embeddings.shape)

print("Training labels:", y_train_framepad.shape)
print("Validation labels:", y_val_framepad.shape)

print(
    "All training features finite:",
    np.isfinite(X_train_embeddings).all()
)

print(
    "All validation features finite:",
    np.isfinite(X_val_embeddings).all()
)


In [ ]:
# Create an explicit input for the feature extractor
feature_input = tf.keras.Input(shape=(64, 157, 3))

x = feature_input

# Pass the input through the trained layers up to Dense(32)
# Layer 8 is dense_66, which produces our 32 learned features.

for layer in patient_balanced_3ch_model.layers[:9]:

    if isinstance(layer, tf.keras.layers.Dropout):
        # Disable dropout during feature extraction
        x = layer(x, training=False)
    else:
        x = layer(x)

# Create a model that outputs the 32-dimensional embedding
feature_extractor = tf.keras.Model(
    inputs=feature_input,
    outputs=x,
    name="patient_balanced_feature_extractor"
)

print("Feature extractor created.")
print("Output shape:", feature_extractor.output_shape)

# Extract training embeddings
X_train_embeddings = feature_extractor.predict(
    X_train_3ch,
    batch_size=32,
    verbose=1
)

# Extract validation embeddings
X_val_embeddings = feature_extractor.predict(
    X_val_3ch,
    batch_size=32,
    verbose=1
)

print("\nFeature extraction completed.")

print("Training embeddings:", X_train_embeddings.shape)
print("Validation embeddings:", X_val_embeddings.shape)

print("Training features finite:",
      np.isfinite(X_train_embeddings).all())

print("Validation features finite:",
      np.isfinite(X_val_embeddings).all())


### 13.5 RBF-SVM on CNN embeddings


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Create the SVM pipeline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

# Train using the learned CNN embeddings
svm_model.fit(
    X_train_embeddings,
    y_train_framepad
)

# Predict validation classes
svm_val_predictions = svm_model.predict(
    X_val_embeddings
)

# Evaluate
print("RBF-SVM RESULTS")
print("=" * 50)

print(
    "Validation Accuracy:",
    round(accuracy_score(
        y_val_framepad,
        svm_val_predictions
    ), 4)
)

print(
    "Validation Macro F1:",
    round(f1_score(
        y_val_framepad,
        svm_val_predictions,
        average="macro",
        zero_division=0
    ), 4)
)

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val_framepad,
    svm_val_predictions,
    labels=[0, 1, 2, 3]
))

print("\nClassification Report:")
print(classification_report(
    y_val_framepad,
    svm_val_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "normal",
        "crackle",
        "wheeze",
        "crackle+wheeze"
    ],
    zero_division=0
))


### 13.6 k-NN on CNN embeddings


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# Standardize embeddings using training data only
knn_scaler = StandardScaler()

X_train_knn = knn_scaler.fit_transform(X_train_embeddings)
X_val_knn = knn_scaler.transform(X_val_embeddings)

# Test different numbers of neighbours
k_values = [3, 5, 7, 9, 11, 15]

knn_results = []

for k in k_values:

    knn_model = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance",
        metric="euclidean"
    )

    # Train the k-NN classifier
    knn_model.fit(
        X_train_knn,
        y_train_framepad
    )

    # Validation predictions
    predictions = knn_model.predict(X_val_knn)

    accuracy = accuracy_score(
        y_val_framepad,
        predictions
    )

    macro_f1 = f1_score(
        y_val_framepad,
        predictions,
        average="macro",
        zero_division=0
    )

    knn_results.append({
        "k": k,
        "Accuracy": accuracy,
        "Macro F1": macro_f1
    })

# Display comparison
knn_results_df = pd.DataFrame(knn_results)

print("k-NN RESULTS")
print("=" * 45)
print(knn_results_df.round(4).to_string(index=False))

# Select best k based on validation Macro F1
best_k = int(
    knn_results_df.loc[
        knn_results_df["Macro F1"].idxmax(),
        "k"
    ]
)

print("\nBest k:", best_k)

# Train final k-NN with selected k
knn_model = KNeighborsClassifier(
    n_neighbors=best_k,
    weights="distance",
    metric="euclidean"
)

knn_model.fit(
    X_train_knn,
    y_train_framepad
)

knn_val_predictions = knn_model.predict(X_val_knn)

print("\nBest k-NN Classification Report:")
print(classification_report(
    y_val_framepad,
    knn_val_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "normal",
        "crackle",
        "wheeze",
        "crackle+wheeze"
    ],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val_framepad,
    knn_val_predictions,
    labels=[0, 1, 2, 3]
))


### 13.7 Bagged k-NN on CNN embeddings


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Base classifier: k-NN with k = 11
base_knn = KNeighborsClassifier(
    n_neighbors=11,
    weights="distance",
    metric="euclidean"
)

# Bagging ensemble with 50 bootstrap classifiers
bagging_knn_model = BaggingClassifier(
    estimator=base_knn,
    n_estimators=50,
    max_samples=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# Train the ensemble on the standardized CNN embeddings
print("Training Bagging k-NN...")

bagging_knn_model.fit(
    X_train_knn,
    y_train_framepad
)

print("Training completed.")

# Predict validation classes
bagging_val_predictions = bagging_knn_model.predict(
    X_val_knn
)

# Evaluate
bagging_accuracy = accuracy_score(
    y_val_framepad,
    bagging_val_predictions
)

bagging_macro_f1 = f1_score(
    y_val_framepad,
    bagging_val_predictions,
    average="macro",
    zero_division=0
)

print("\nBAGGING k-NN RESULTS")
print("=" * 50)

print(f"Accuracy: {bagging_accuracy:.4f}")
print(f"Macro F1: {bagging_macro_f1:.4f}")

print("\nConfusion Matrix:")

print(confusion_matrix(
    y_val_framepad,
    bagging_val_predictions,
    labels=[0, 1, 2, 3]
))

print("\nClassification Report:")

print(classification_report(
    y_val_framepad,
    bagging_val_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "normal",
        "crackle",
        "wheeze",
        "crackle+wheeze"
    ],
    zero_division=0
))


### 13.8 Compare embedding-based classifiers


In [ ]:
# Extract 32-dimensional features from the weighted CNN

feature_input_weighted = tf.keras.Input(
    shape=(64, 157, 3)
)

x = feature_input_weighted

# Use all trained layers up to the 32-unit Dense layer
for layer in weighted_3ch_model.layers[:9]:

    if isinstance(layer, tf.keras.layers.Dropout):
        x = layer(x, training=False)
    else:
        x = layer(x)

weighted_feature_extractor = tf.keras.Model(
    inputs=feature_input_weighted,
    outputs=x,
    name="weighted_cnn_feature_extractor"
)

print("Weighted feature extractor created.")
print("Output shape:", weighted_feature_extractor.output_shape)

# Extract training features
X_train_weighted_embeddings = (
    weighted_feature_extractor.predict(
        X_train_3ch,
        batch_size=32,
        verbose=1
    )
)

# Extract validation features
X_val_weighted_embeddings = (
    weighted_feature_extractor.predict(
        X_val_3ch,
        batch_size=32,
        verbose=1
    )
)

print("\nFeature extraction completed.")

print(
    "Training embeddings:",
    X_train_weighted_embeddings.shape
)

print(
    "Validation embeddings:",
    X_val_weighted_embeddings.shape
)

print(
    "All features finite:",
    np.isfinite(X_train_weighted_embeddings).all()
    and np.isfinite(X_val_weighted_embeddings).all()
)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# Standardize weighted CNN embeddings
weighted_knn_scaler = StandardScaler()

X_train_weighted_knn = weighted_knn_scaler.fit_transform(
    X_train_weighted_embeddings
)

X_val_weighted_knn = weighted_knn_scaler.transform(
    X_val_weighted_embeddings
)

# Try the same k values as the previous experiment
k_values = [3, 5, 7, 9, 11, 15]

weighted_knn_results = []

for k in k_values:

    weighted_knn_model = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance",
        metric="euclidean"
    )

    weighted_knn_model.fit(
        X_train_weighted_knn,
        y_train_framepad
    )

    predictions = weighted_knn_model.predict(
        X_val_weighted_knn
    )

    accuracy = accuracy_score(
        y_val_framepad,
        predictions
    )

    macro_f1 = f1_score(
        y_val_framepad,
        predictions,
        average="macro",
        zero_division=0
    )

    weighted_knn_results.append({
        "k": k,
        "Accuracy": accuracy,
        "Macro F1": macro_f1
    })

# Display results
weighted_knn_results_df = pd.DataFrame(
    weighted_knn_results
)

print("WEIGHTED CNN + k-NN RESULTS")
print("=" * 50)

print(
    weighted_knn_results_df.round(4).to_string(index=False)
)

# Select best k using validation Macro F1
best_weighted_k = int(
    weighted_knn_results_df.loc[
        weighted_knn_results_df["Macro F1"].idxmax(),
        "k"
    ]
)

print("\nBest k:", best_weighted_k)

# Train selected model
weighted_knn_model = KNeighborsClassifier(
    n_neighbors=best_weighted_k,
    weights="distance",
    metric="euclidean"
)

weighted_knn_model.fit(
    X_train_weighted_knn,
    y_train_framepad
)

weighted_knn_val_predictions = weighted_knn_model.predict(
    X_val_weighted_knn
)

print("\nBEST WEIGHTED CNN + k-NN REPORT")
print("=" * 50)

print(classification_report(
    y_val_framepad,
    weighted_knn_val_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "normal",
        "crackle",
        "wheeze",
        "crackle+wheeze"
    ],
    zero_division=0
))

print("\nConfusion Matrix:")

print(confusion_matrix(
    y_val_framepad,
    weighted_knn_val_predictions,
    labels=[0, 1, 2, 3]
))


## 14. CNN–LSTM temporal model


The CNN–LSTM combines local spectral feature extraction with temporal modelling. The selected model produces independent crackle and wheeze event scores, which are later mapped to the four acoustic categories.


### 14.1 Define the CNN–LSTM architecture


In [ ]:
# Lightweight CNN-LSTM for respiratory event classification

from tensorflow.keras import layers, Model

cnn_lstm_input = layers.Input(
    shape=(64, 157, 3),
    name="lung_sound_input"
)

# CNN feature extraction
x = layers.Conv2D(
    16, (3, 3), padding="same", activation="relu"
)(cnn_lstm_input)

x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(
    32, (3, 3), padding="same", activation="relu"
)(x)

x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(
    64, (3, 3), padding="same", activation="relu"
)(x)

# Reduce frequency dimension while retaining time dimension
x = layers.AveragePooling2D(
    pool_size=(4, 1)
)(x)

# Convert feature maps into a time sequence
# Expected before Permute: (4, 39, 64)
x = layers.Permute((2, 1, 3))(x)

# Expected sequence: (39 time steps, 256 features)
x = layers.Reshape((39, 256))(x)

# Temporal feature learning
x = layers.LSTM(
    64,
    return_sequences=False,
    name="temporal_lstm"
)(x)

x = layers.Dropout(0.3)(x)

# Classification head
x = layers.Dense(
    32,
    activation="relu",
    name="cnn_lstm_dense"
)(x)

x = layers.Dropout(0.2)(x)

cnn_lstm_output = layers.Dense(
    2,
    activation="sigmoid",
    name="event_predictions"
)(x)

cnn_lstm_model = Model(
    inputs=cnn_lstm_input,
    outputs=cnn_lstm_output,
    name="lightweight_cnn_lstm"
)

cnn_lstm_model.summary()

print(
    "\nOutput shape:",
    cnn_lstm_model.output_shape
)


## 15. Official test evaluation and error analysis


The official test subset is evaluated only after the validation operating thresholds are frozen. The analysis then moves from aggregate metrics to patient, recording, class, and device failure modes.


### 15.1 Reconstruct the test feature contract


In [ ]:
# ============================================================
# ============================================================

import numpy as np
import pandas as pd
import librosa


def build_test_logmel_with_masks(
    frame,
    split_name,
    target_frames=157
):
    features = []
    labels = []
    frame_masks = []
    metadata = []

    invalid_segments = 0
    audio_cropped_segments = 0
    frame_padded_segments = 0
    frame_cropped_segments = 0

    # Follow the exact recording-group order of the selected
    # build_features_framepad() training/validation pipeline.
    for recording_number, (wav_path, group) in enumerate(
        frame.groupby("wav_path", sort=False),
        start=1
    ):

        # Load each recording only once
        audio, sr = librosa.load(
            wav_path,
            sr=TARGET_SR,
            mono=True
        )

        for _, row in group.iterrows():

            # Match the original Log-Mel builder's indexing
            start_idx = int(np.floor(row["start"] * sr))
            end_idx = int(np.ceil(row["end"] * sr))

            start_idx = max(0, start_idx)
            end_idx = min(len(audio), end_idx)

            if end_idx <= start_idx:
                invalid_segments += 1
                continue

            segment = audio[start_idx:end_idx]

            # Crop long audio segments to five seconds
            if len(segment) > TARGET_SAMPLES:
                audio_cropped_segments += 1

                segment = fit_audio_length(
                    segment,
                    TARGET_SAMPLES
                )

            # Extract Log-Mel before time-frame padding/cropping
            log_mel = extract_logmel(segment)

            source_frames = log_mel.shape[1]

            # Construct a mask of genuine frames
            frame_mask = np.zeros(
                target_frames,
                dtype=bool
            )

            if source_frames < target_frames:

                frame_padded_segments += 1

                left_padding = (
                    target_frames - source_frames
                ) // 2

                frame_mask[
                    left_padding:left_padding + source_frames
                ] = True

            else:

                if source_frames > target_frames:
                    frame_cropped_segments += 1

                frame_mask[:] = True

            # Apply the same centre padding/cropping
            log_mel = fit_logmel_frames(
                log_mel,
                target_frames=target_frames
            )

            features.append(
                log_mel[..., np.newaxis]
            )

            label = str(row["label"])

            if label not in class_to_idx:
                raise ValueError(
                    f"Unknown label: {label}"
                )

            labels.append(class_to_idx[label])
            frame_masks.append(frame_mask)

            metadata.append({
                "wav_path": str(wav_path),
                "recording": row["recording"],
                "patient": row["patient"],
                "start": float(row["start"]),
                "end": float(row["end"]),
                "label": label,
                "label_id": class_to_idx[label]
            })

        if recording_number % 100 == 0:
            print(
                f"{split_name}: processed "
                f"{recording_number} recordings"
            )

    if not features:
        raise ValueError("No test features generated.")

    X = np.stack(features).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    masks = np.stack(frame_masks)

    metadata_df = pd.DataFrame(metadata)

    print(f"\n{split_name} feature extraction completed.")
    print("Features:", X.shape)
    print("Labels:", y.shape)
    print("Frame masks:", masks.shape)
    print("Metadata:", metadata_df.shape)
    print("Invalid segments:", invalid_segments)
    print("Audio segments cropped:", audio_cropped_segments)
    print("Spectrograms padded:", frame_padded_segments)
    print("Spectrograms cropped:", frame_cropped_segments)
    print(
        "Feature memory (MB):",
        round(X.nbytes / (1024 ** 2), 2)
    )

    return X, y, masks, metadata_df


# ------------------------------------------------------------
# Extract official test Log-Mel features and frame masks
# ------------------------------------------------------------

(
    X_test_framepad,
    y_test_framepad,
    test_frame_mask,
    test_feature_metadata
) = build_test_logmel_with_masks(
    meta_test,
    split_name="Official Test",
    target_frames=157
)


# ------------------------------------------------------------
# Verify feature order and label alignment
# ------------------------------------------------------------

# Reconstruct the same grouped order used by the feature builder.
test_feature_order_df = pd.concat(
    [
        group
        for _, group in meta_test.groupby(
            "wav_path",
            sort=False
        )
    ],
    ignore_index=True
)

assert len(X_test_framepad) == len(meta_test) == 2756

assert np.array_equal(
    test_feature_order_df["label"].map(class_to_idx).to_numpy(),
    y_test_framepad
)

for col in ["recording", "patient", "label"]:
    assert np.array_equal(
        test_feature_metadata[col].astype(str).to_numpy(),
        test_feature_order_df[col].astype(str).to_numpy()
    ), f"Metadata alignment failed: {col}"

for col in ["start", "end"]:
    assert np.allclose(
        test_feature_metadata[col].to_numpy(),
        test_feature_order_df[col].to_numpy()
    ), f"Metadata alignment failed: {col}"

print("\nTest labels and metadata alignment verified.")


# ------------------------------------------------------------
# Build the three-channel features
# ------------------------------------------------------------

X_test_3ch = build_three_channel_features(
    X_test_framepad,
    test_frame_mask,
    "Official Test"
)


# ------------------------------------------------------------
# Apply TRAINING-ONLY delta normalization statistics
# ------------------------------------------------------------

for channel in [1, 2]:

    X_test_3ch[..., channel] -= (
        delta_means[channel][None, :, None]
    )

    X_test_3ch[..., channel] /= (
        delta_stds[channel][None, :, None]
    )

    # Restore padded frames to zero
    X_test_3ch[..., channel] *= (
        test_frame_mask[:, None, :]
    )


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert X_test_3ch.shape == (2756, 64, 157, 3)
assert y_test_framepad.shape == (2756,)
assert test_frame_mask.shape == (2756, 157)

assert np.isfinite(X_test_3ch).all()
assert np.isfinite(y_test_framepad).all()

for channel in [1, 2]:

    padded_values = np.where(
        test_frame_mask[:, None, :],
        0.0,
        X_test_3ch[..., channel]
    )

    assert np.max(np.abs(padded_values)) < 1e-6

print("\n" + "=" * 60)
print("OFFICIAL TEST FEATURES READY")
print("=" * 60)

print("X_test_3ch:", X_test_3ch.shape)
print("y_test_framepad:", y_test_framepad.shape)
print("Test frame mask:", test_frame_mask.shape)
print("All values finite:", np.isfinite(X_test_3ch).all())
print("Patient count:", test_feature_metadata["patient"].nunique())

print("\nTest class distribution:")

print(
    pd.Series(
        y_test_framepad
    ).map(idx_to_class).value_counts()
)


### 15.2 Generate event scores and four-class predictions


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# FROZEN VALIDATION THRESHOLDS
# ============================================================

CRACKLE_THRESHOLD = 0.45
WHEEZE_THRESHOLD = 0.60

print("Frozen thresholds:")
print("Crackle:", CRACKLE_THRESHOLD)
print("Wheeze:", WHEEZE_THRESHOLD)


# ============================================================
# PREDICT ON OFFICIAL TEST DATA
# ============================================================

cnn_lstm_test_prob = cnn_lstm_model.predict(
    X_test_3ch,
    batch_size=32,
    verbose=1
)

assert cnn_lstm_test_prob.shape == (2756, 2)
assert np.isfinite(cnn_lstm_test_prob).all()

print("\nTest probability shape:", cnn_lstm_test_prob.shape)


# ============================================================
# CREATE MULTI-LABEL TEST TARGETS
# ============================================================

# Output 0 = crackle
# Output 1 = wheeze

y_test_multi = np.column_stack([
    np.isin(
        y_test_framepad,
        [
            class_to_idx["crackle"],
            class_to_idx["crackle+wheeze"]
        ]
    ),
    np.isin(
        y_test_framepad,
        [
            class_to_idx["wheeze"],
            class_to_idx["crackle+wheeze"]
        ]
    )
]).astype(np.int32)

print("Test multi-label shape:", y_test_multi.shape)


# ============================================================
# EVENT-WISE PERFORMANCE
# ============================================================

event_names = ["Crackle", "Wheeze"]
event_thresholds = [
    CRACKLE_THRESHOLD,
    WHEEZE_THRESHOLD
]

for i, event in enumerate(event_names):

    y_true = y_test_multi[:, i]
    y_prob = cnn_lstm_test_prob[:, i]
    threshold = event_thresholds[i]

    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0
        )
    )

    print("\n" + "=" * 55)
    print(event.upper(), "TEST RESULTS")
    print("=" * 55)

    print(f"ROC-AUC:           {auc:.4f}")
    print(f"Average Precision: {ap:.4f}")
    print(f"Threshold:         {threshold:.2f}")
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1-score:          {f1:.4f}")
    print(f"Actual positives:  {y_true.sum()}")
    print(f"Predicted positive:{y_pred.sum()}")

    print("\nBinary confusion matrix:")
    print(confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ))


# ============================================================
# CONVERT TWO EVENT PREDICTIONS INTO FOUR CLASSES
# ============================================================

pred_crackle = (
    cnn_lstm_test_prob[:, 0] >= CRACKLE_THRESHOLD
)

pred_wheeze = (
    cnn_lstm_test_prob[:, 1] >= WHEEZE_THRESHOLD
)

cnn_lstm_test_predictions = np.select(
    [
        (~pred_crackle) & (~pred_wheeze),
        pred_crackle & (~pred_wheeze),
        (~pred_crackle) & pred_wheeze,
        pred_crackle & pred_wheeze
    ],
    [
        class_to_idx["normal"],
        class_to_idx["crackle"],
        class_to_idx["wheeze"],
        class_to_idx["crackle+wheeze"]
    ],
    default=class_to_idx["normal"]
)


# ============================================================
# FOUR-CLASS TEST PERFORMANCE
# ============================================================

class_names = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

class_ids = [
    class_to_idx[name]
    for name in class_names
]

test_accuracy = accuracy_score(
    y_test_framepad,
    cnn_lstm_test_predictions
)

test_macro_f1 = f1_score(
    y_test_framepad,
    cnn_lstm_test_predictions,
    labels=class_ids,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test_framepad,
    cnn_lstm_test_predictions,
    labels=class_ids,
    average="weighted",
    zero_division=0
)

test_cm = confusion_matrix(
    y_test_framepad,
    cnn_lstm_test_predictions,
    labels=class_ids
)

print("\n" + "=" * 60)
print("OFFICIAL TEST: FOUR-CLASS RESULTS")
print("=" * 60)

print(f"Accuracy:    {test_accuracy:.4f}")
print(f"Macro F1:    {test_macro_f1:.4f}")
print(f"Weighted F1: {test_weighted_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nClassification Report:")

print(classification_report(
    y_test_framepad,
    cnn_lstm_test_predictions,
    labels=class_ids,
    target_names=class_names,
    zero_division=0
))


# ============================================================
# ICBHI SCORE USING YOUR NOTEBOOK'S DEFINITION
# ============================================================

normal_specificity = (
    test_cm[0, 0] / test_cm[0, :].sum()
)

abnormal_sensitivity = (
    np.trace(test_cm[1:, 1:]) /
    test_cm[1:, :].sum()
)

test_icbhi_score = (
    normal_specificity + abnormal_sensitivity
) / 2

print("\nICBHI-style evaluation:")
print(f"Normal specificity:   {normal_specificity:.4f}")
print(f"Abnormal sensitivity: {abnormal_sensitivity:.4f}")
print(f"ICBHI score:          {test_icbhi_score:.4f}")


# ============================================================
# PREDICTED CLASS DISTRIBUTION
# ============================================================

print("\nPredicted class distribution:")

for class_id, class_name in zip(class_ids, class_names):

    count = np.sum(
        cnn_lstm_test_predictions == class_id
    )

    percentage = (
        count / len(cnn_lstm_test_predictions) * 100
    )

    print(
        f"{class_name:18s}: "
        f"{count:5d} ({percentage:.2f}%)"
    )


# ============================================================
# SAVE TEST PREDICTIONS FOR PROJECT RECORDS
# ============================================================

test_predictions_df = test_feature_metadata.copy()

test_predictions_df["actual_class"] = [
    idx_to_class[int(x)]
    for x in y_test_framepad
]

test_predictions_df["predicted_class"] = [
    idx_to_class[int(x)]
    for x in cnn_lstm_test_predictions
]

test_predictions_df["crackle_probability"] = (
    cnn_lstm_test_prob[:, 0]
)

test_predictions_df["wheeze_probability"] = (
    cnn_lstm_test_prob[:, 1]
)

test_predictions_df.to_csv(
    "cnn_lstm_official_test_predictions.csv",
    index=False
)

print("\nTest predictions saved successfully.")
print("File: cnn_lstm_official_test_predictions.csv")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

class_names = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

class_ids = [0, 1, 2, 3]

patient_results = []

for patient, group in test_predictions_df.groupby("patient"):

    y_true = group["actual_class"].map(class_to_idx).to_numpy()
    y_pred = group["predicted_class"].map(class_to_idx).to_numpy()

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_ids
    )

    normal_recall = (
        cm[0, 0] / cm[0, :].sum()
        if cm[0, :].sum() > 0 else np.nan
    )

    abnormal_total = cm[1:, :].sum()

    abnormal_sensitivity = (
        np.trace(cm[1:, 1:]) / abnormal_total
        if abnormal_total > 0 else np.nan
    )

    patient_results.append({
        "Patient": str(patient),
        "Cycles": len(group),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            labels=class_ids,
            average="macro",
            zero_division=0
        ),
        "Normal recall": normal_recall,
        "Abnormal sensitivity": abnormal_sensitivity
    })

patient_results_df = pd.DataFrame(patient_results)

print("PATIENT-WISE TEST PERFORMANCE")
print("=" * 65)

print("Patients evaluated:", len(patient_results_df))

display(
    patient_results_df.sort_values(
        "Cycles",
        ascending=False
    ).round(3)
)

print("\nOverall patient-level summary:")
display(
    patient_results_df[
        ["Accuracy", "Macro F1", "Normal recall", "Abnormal sensitivity"]
    ].describe().round(3)
)


In [ ]:
class_order = ["normal", "crackle", "wheeze", "crackle+wheeze"]

worst_patients = [185, 187, 214]

for patient_id in worst_patients:
    patient_data = test_predictions_df[
        test_predictions_df["patient"] == patient_id
    ]

    print(f"\nPatient {patient_id} — {len(patient_data)} cycles")

    comparison = pd.crosstab(
        patient_data["actual_class"],
        patient_data["predicted_class"]
    )

    comparison = comparison.reindex(
        index=class_order,
        columns=class_order,
        fill_value=0
    )

    display(comparison)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

class_order = ["normal", "crackle", "wheeze", "crackle+wheeze"]

patient_metrics = []

for patient_id, group in test_predictions_df.groupby("patient"):

    y_true = group["actual_class"].astype(str)
    y_pred = group["predicted_class"].astype(str)

    normal_mask = y_true == "normal"
    abnormal_mask = y_true != "normal"

    normal_recall = (
        ((y_true == "normal") & (y_pred == "normal")).sum()
        / normal_mask.sum()
        if normal_mask.sum() > 0 else float("nan")
    )

    abnormal_sensitivity = (
        ((y_true != "normal") & (y_pred == y_true)).sum()
        / abnormal_mask.sum()
        if abnormal_mask.sum() > 0 else float("nan")
    )

    patient_metrics.append({
        "patient": patient_id,
        "cycles": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred,
            labels=class_order,
            average="macro",
            zero_division=0
        ),
        "normal_recall": normal_recall,
        "abnormal_sensitivity": abnormal_sensitivity
    })

patient_metrics_df = pd.DataFrame(patient_metrics)

print("10 lowest Macro F1 patient scores:")
display(
    patient_metrics_df
    .sort_values("macro_f1")
    .head(10)
    .round(3)
)

print("\nRecalculated results for patients 185, 187 and 214:")
display(
    patient_metrics_df[
        patient_metrics_df["patient"].isin(["185", "187", "214"])
    ].round(3)
)


### 15.3 Inspect device coverage on the test subset


In [ ]:
device_test_df = test_predictions_df.copy()

# Extract the device name from the end of each recording name
device_test_df["device"] = (
    device_test_df["recording"]
    .astype(str)
    .str.rsplit("_", n=1)
    .str[-1]
)

# Mark whether each cycle was classified correctly
device_test_df["correct"] = (
    device_test_df["actual_class"] ==
    device_test_df["predicted_class"]
)

# Summary of the number of recordings, patients and cycles per device
print("DEVICE DISTRIBUTION")

device_counts = (
    device_test_df.groupby("device")
    .agg(
        patients=("patient", "nunique"),
        recordings=("recording", "nunique"),
        cycles=("recording", "size")
    )
    .reset_index()
)

display(device_counts)

# Compare model behaviour for each actual class on each device
device_class_summary = (
    device_test_df.groupby(["device", "actual_class"])
    .agg(
        cycles=("actual_class", "size"),
        accuracy=("correct", "mean"),
        mean_crackle=("crackle_probability", "mean"),
        mean_wheeze=("wheeze_probability", "mean"),
        crackle_threshold_rate=(
            "crackle_probability",
            lambda x: (x >= 0.45).mean()
        ),
        wheeze_threshold_rate=(
            "wheeze_probability",
            lambda x: (x >= 0.60).mean()
        )
    )
    .reset_index()
)

print("\nPERFORMANCE BY DEVICE AND ACTUAL CLASS")
display(device_class_summary.round(3))


### 15.4 Compare class and device distributions


In [ ]:
split_frames = {
    "Training": meta_train_framepad,
    "Validation": meta_val_framepad,
    "Official Test": meta_test
}

device_summaries = []
class_summaries = []

for split_name, frame in split_frames.items():

    temp = frame.copy()

    # Extract device name from the recording filename
    temp["device"] = (
        temp["recording"]
        .astype(str)
        .str.rsplit("_", n=1)
        .str[-1]
    )

    # Identify the actual class column
    class_col = "label" if "label" in temp.columns else "actual_class"

    # Overall device distribution
    summary = (
        temp.groupby("device")
        .agg(
            patients=("patient", "nunique"),
            recordings=("recording", "nunique"),
            cycles=("recording", "size")
        )
        .reset_index()
    )

    summary.insert(0, "split", split_name)
    device_summaries.append(summary)

    # Class distribution within each device
    class_summary = (
        temp.groupby(["device", class_col])
        .size()
        .reset_index(name="cycles")
    )

    class_summary["class_percentage"] = (
        class_summary["cycles"]
        / class_summary.groupby("device")["cycles"].transform("sum")
        * 100
    )

    class_summary.insert(0, "split", split_name)
    class_summaries.append(class_summary)


print("DEVICE DISTRIBUTION ACROSS SPLITS")

device_distribution_df = pd.concat(
    device_summaries, ignore_index=True
)

display(device_distribution_df)

print("\nCLASS DISTRIBUTION WITHIN EACH DEVICE")

device_class_distribution_df = pd.concat(
    class_summaries, ignore_index=True
)

display(device_class_distribution_df.round(2))


### 15.5 Analyze errors within recordings


In [ ]:
def get_device(frame):
    temp = frame.copy()
    temp["device"] = (
        temp["recording"]
        .astype(str)
        .str.rsplit("_", n=1)
        .str[-1]
    )
    return temp

train_device_data = get_device(meta_train_framepad)
val_device_data = get_device(meta_val_framepad)

litt_train = train_device_data[
    train_device_data["device"] == "Litt3200"
]

litt_val = val_device_data[
    val_device_data["device"] == "Litt3200"
]

print("Litt3200 training cycles:", len(litt_train))
print("Litt3200 training patients:", litt_train["patient"].nunique())

print("\nTraining distribution by patient and class:")

display(
    litt_train.groupby(["patient", "label"])
    .size()
    .reset_index(name="cycles")
)

print("\nLitt3200 validation cycles:", len(litt_val))


In [ ]:
litt_test = test_predictions_df.copy()

litt_test["device"] = (
    litt_test["recording"]
    .astype(str)
    .str.rsplit("_", n=1)
    .str[-1]
)

litt_test = litt_test[
    litt_test["device"] == "Litt3200"
].copy()

patient_results = []

class_order = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

for patient_id, group in litt_test.groupby("patient"):

    result = {
        "patient": patient_id,
        "cycles": len(group),
        "recordings": group["recording"].nunique(),
        "accuracy": (
            group["actual_class"] == group["predicted_class"]
        ).mean()
    }

    for cls in class_order:
        actual = group["actual_class"] == cls
        support = actual.sum()

        result[f"{cls}_cycles"] = support

        result[f"{cls}_recall"] = (
            (
                group.loc[actual, "predicted_class"] == cls
            ).mean()
            if support > 0 else np.nan
        )

    patient_results.append(result)

litt_patient_results = pd.DataFrame(patient_results)

print("Litt3200 test patients:", len(litt_patient_results))

display(
    litt_patient_results
    .sort_values("cycles", ascending=False)
    .round(3)
)


### 15.6 Rank low-performing recordings


In [ ]:
class_order = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

recording_results = []

for (patient_id, recording), group in litt_test.groupby(
    ["patient", "recording"]
):

    y_true = group["actual_class"]
    y_pred = group["predicted_class"]

    row = {
        "patient": patient_id,
        "recording": recording,
        "cycles": len(group),
        "accuracy": (y_true == y_pred).mean()
    }

    for cls in class_order:
        mask = y_true == cls
        count = mask.sum()

        row[f"{cls}_count"] = count

        row[f"{cls}_recall"] = (
            (y_pred[mask] == cls).mean()
            if count > 0 else np.nan
        )

    recording_results.append(row)

litt_recording_results = pd.DataFrame(recording_results)

print("Total Litt3200 recordings:", len(litt_recording_results))

print("\n15 recordings with lowest accuracy:")

display(
    litt_recording_results
    .sort_values("accuracy")
    .head(15)
    .round(3)
)


In [ ]:
class_order = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

# Select the 15 recordings with the lowest accuracy
worst_recordings = (
    litt_recording_results
    .sort_values("accuracy")
    .head(15)["recording"]
)

rows = []

for recording in worst_recordings:

    group = litt_test[litt_test["recording"] == recording]

    row = {
        "patient": group["patient"].iloc[0],
        "recording": recording,
        "cycles": len(group),
        "accuracy": (
            group["actual_class"] == group["predicted_class"]
        ).mean()
    }

    # Count actual and predicted classes separately
    for cls in class_order:
        row[f"actual_{cls}"] = (
            group["actual_class"] == cls
        ).sum()

        row[f"predicted_{cls}"] = (
            group["predicted_class"] == cls
        ).sum()

    rows.append(row)

worst_recording_errors = pd.DataFrame(rows)

display(worst_recording_errors.round(3))


In [ ]:
class_order = [
    "normal",
    "crackle",
    "wheeze",
    "crackle+wheeze"
]

# Use the original prediction dataframe
litt_test = test_predictions_df.copy()

litt_test["device"] = (
    litt_test["recording"]
    .astype(str)
    .str.rsplit("_", n=1)
    .str[-1]
)

litt_test = litt_test[
    litt_test["device"] == "Litt3200"
].copy()

# Actual class counts for every recording
actual_counts = pd.crosstab(
    litt_test["recording"],
    litt_test["actual_class"]
).reindex(columns=class_order, fill_value=0)

# Predicted class counts for every recording
predicted_counts = pd.crosstab(
    litt_test["recording"],
    litt_test["predicted_class"]
).reindex(columns=class_order, fill_value=0)

# Summary of recordings
recording_audit = (
    litt_test.groupby("recording")
    .agg(
        patient=("patient", "first"),
        cycles=("recording", "size"),
        accuracy=(
            "actual_class",
            lambda x: (
                x == litt_test.loc[x.index, "predicted_class"]
            ).mean()
        )
    )
)

for cls in class_order:
    recording_audit[f"actual_{cls}"] = actual_counts[cls]
    recording_audit[f"predicted_{cls}"] = predicted_counts[cls]

recording_audit["actual_sum"] = (
    recording_audit[[f"actual_{c}" for c in class_order]].sum(axis=1)
)

recording_audit["predicted_sum"] = (
    recording_audit[[f"predicted_{c}" for c in class_order]].sum(axis=1)
)

recording_audit["count_check"] = (
    (recording_audit["actual_sum"] == recording_audit["cycles"]) &
    (recording_audit["predicted_sum"] == recording_audit["cycles"])
)

print("Total recordings:", len(recording_audit))
print("All count checks passed:", recording_audit["count_check"].all())

display(
    recording_audit
    .sort_values("accuracy")
    .head(15)
    .round(3)
)


## 16. Device-stratified validation analysis


### 16.1 Patient-level validation behaviour


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

patient_results = []

for patient, indices in candidate_val_df.groupby("patient").groups.items():

    indices = np.asarray(list(indices), dtype=int)

    row = {
        "patient": patient,
        "samples": len(indices),
        "crackle_positive": int(y_val_multi[indices, 0].sum()),
        "wheeze_positive": int(y_val_multi[indices, 1].sum())
    }

    for i, event in enumerate(["crackle", "wheeze"]):

        y_true = y_val_multi[indices, i].astype(int)
        y_prob = cnn_lstm_val_prob[indices, i]

        if len(np.unique(y_true)) == 2:
            row[f"{event}_auc"] = roc_auc_score(y_true, y_prob)
            row[f"{event}_ap"] = average_precision_score(y_true, y_prob)
        else:
            row[f"{event}_auc"] = np.nan
            row[f"{event}_ap"] = np.nan

    patient_results.append(row)

patient_results_df = pd.DataFrame(patient_results)

print(patient_results_df.sort_values("samples", ascending=False).to_string(index=False))


In [ ]:
val_feature_order_df = pd.concat(
    [
        group
        for _, group in candidate_val_df.groupby(
            "wav_path", sort=False
        )
    ],
    ignore_index=True
)

print("Feature samples:", len(X_val_3ch))
print("Metadata samples:", len(val_feature_order_df))

assert len(val_feature_order_df) == len(X_val_3ch)

print(
    "Framepad labels aligned:",
    np.array_equal(
        val_feature_order_df["label"]
        .map(class_to_idx)
        .to_numpy(),
        y_val_framepad
    )
)

expected_multi = np.column_stack([
    (y_val_framepad == class_to_idx["crackle"]) |
    (y_val_framepad == class_to_idx["crackle+wheeze"]),

    (y_val_framepad == class_to_idx["wheeze"]) |
    (y_val_framepad == class_to_idx["crackle+wheeze"])
]).astype(np.int32)

print(
    "Multi-label targets aligned:",
    np.array_equal(expected_multi, y_val_multi)
)


### 16.2 Device-level event ranking


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

device_results = []

for device, group in val_feature_order_df.groupby("device"):

    indices = group.index.to_numpy()

    row = {
        "device": device,
        "samples": len(indices),
        "patients": group["patient"].nunique()
    }

    for i, event in enumerate(["crackle", "wheeze"]):

        y_true = y_val_multi[indices, i].astype(int)
        y_prob = cnn_lstm_val_prob[indices, i]

        if len(np.unique(y_true)) == 2:
            row[f"{event}_auc"] = roc_auc_score(y_true, y_prob)
            row[f"{event}_ap"] = average_precision_score(y_true, y_prob)
        else:
            row[f"{event}_auc"] = np.nan
            row[f"{event}_ap"] = np.nan

    device_results.append(row)

print(pd.DataFrame(device_results).to_string(index=False))


In [ ]:
device_class_summary = (
    candidate_val_df
    .groupby("device")
    .agg(
        samples=("label", "size"),
        patients=("patient", "nunique"),
        crackle_positive=("crackle", "sum"),
        wheeze_positive=("wheeze", "sum")
    )
)

device_class_summary["crackle_negative"] = (
    device_class_summary["samples"]
    - device_class_summary["crackle_positive"]
)

device_class_summary["wheeze_negative"] = (
    device_class_summary["samples"]
    - device_class_summary["wheeze_positive"]
)

print(device_class_summary)


## 17. Device alignment experiment


A training-derived spectral correction is evaluated as a device-variation diagnostic. The correction is fitted from training data and applied without using the true class of a future test segment.


### 17.1 Apply training-derived device corrections


In [ ]:
X_train_3ch_aligned = build_three_channel_features(
    X_train_logmel_aligned,
    train_mfcc_mask,
    "Aligned Train"
)

X_val_3ch_aligned = build_three_channel_features(
    X_val_logmel_aligned,
    val_mfcc_mask,
    "Aligned Validation"
)

print("\nFinal shapes:")
print("Train:", X_train_3ch_aligned.shape)
print("Validation:", X_val_3ch_aligned.shape)


### 17.2 Compare original and aligned feature channels


In [ ]:
def compare_channels(X_original, X_aligned, name):

    print(f"\n========== {name} ==========")

    for ch, channel_name in enumerate([
        "Log-Mel",
        "Delta",
        "Delta-Delta"
    ]):

        original = X_original[..., ch]
        aligned = X_aligned[..., ch]

        print(f"\n{channel_name}")

        print(
            "Original:",
            "mean =", round(float(original.mean()), 5),
            "std =", round(float(original.std()), 5),
            "min =", round(float(original.min()), 5),
            "max =", round(float(original.max()), 5)
        )

        print(
            "Aligned:",
            "mean =", round(float(aligned.mean()), 5),
            "std =", round(float(aligned.std()), 5),
            "min =", round(float(aligned.min()), 5),
            "max =", round(float(aligned.max()), 5)
        )

        print(
            "Fraction at zero:",
            round(float(np.mean(aligned == 0)), 4)
        )

        if ch == 0:
            print(
                "Fraction at upper clipping limit:",
                round(float(np.mean(aligned == 1)), 4)
            )


compare_channels(
    X_train_3ch,
    X_train_3ch_aligned,
    "TRAIN"
)

compare_channels(
    X_val_3ch,
    X_val_3ch_aligned,
    "VALIDATION"
)


### 17.3 Reconstruct aligned derivative channels


In [ ]:
X_train_3ch_aligned_raw = build_three_channel_features(
    X_train_logmel_aligned,
    train_mfcc_mask,
    "Aligned Train"
)

X_val_3ch_aligned_raw = build_three_channel_features(
    X_val_logmel_aligned,
    val_mfcc_mask,
    "Aligned Validation"
)

X_train_3ch_aligned = X_train_3ch_aligned_raw.copy()
X_val_3ch_aligned = X_val_3ch_aligned_raw.copy()

for channel in [1, 2]:

    mean = delta_means[channel]
    std = delta_stds[channel]

    X_train_3ch_aligned[..., channel] = (
        X_train_3ch_aligned[..., channel]
        - mean[None, :, None]
    ) / std[None, :, None]

    X_val_3ch_aligned[..., channel] = (
        X_val_3ch_aligned[..., channel]
        - mean[None, :, None]
    ) / std[None, :, None]

    # Preserve padding
    X_train_3ch_aligned[..., channel] *= (
        train_mfcc_mask[:, None, :]
    )

    X_val_3ch_aligned[..., channel] *= (
        val_mfcc_mask[:, None, :]
    )

print("Train:", X_train_3ch_aligned.shape)
print("Validation:", X_val_3ch_aligned.shape)

print("Train finite:", np.isfinite(X_train_3ch_aligned).all())
print("Validation finite:", np.isfinite(X_val_3ch_aligned).all())


### 17.4 Compare baseline and aligned validation predictions


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

# Predictions using the same existing checkpoint
original_val_prob = cnn_lstm_model.predict(
    X_val_3ch,
    batch_size=32,
    verbose=1
)

aligned_val_prob = cnn_lstm_model.predict(
    X_val_3ch_aligned,
    batch_size=32,
    verbose=1
)

print("Original predictions:", original_val_prob.shape)
print("Aligned predictions:", aligned_val_prob.shape)

# Compare threshold-independent metrics
comparison = []

for i, event in enumerate(["Crackle", "Wheeze"]):

    y_true = y_val_multi[:, i].astype(int)

    original_auc = roc_auc_score(y_true, original_val_prob[:, i])
    original_ap = average_precision_score(y_true, original_val_prob[:, i])

    aligned_auc = roc_auc_score(y_true, aligned_val_prob[:, i])
    aligned_ap = average_precision_score(y_true, aligned_val_prob[:, i])

    comparison.append({
        "Event": event,
        "Original ROC-AUC": original_auc,
        "Aligned ROC-AUC": aligned_auc,
        "ROC-AUC change": aligned_auc - original_auc,
        "Original AP": original_ap,
        "Aligned AP": aligned_ap,
        "AP change": aligned_ap - original_ap
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df.round(4))


### 17.5 Patient-level bootstrap diagnostic


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_bootstrap = 2000

patient_ids = val_feature_order_df["patient"].to_numpy()
unique_patients = np.unique(patient_ids)

# Preserve all cycles belonging to each patient
patient_indices = {
    p: np.where(patient_ids == p)[0]
    for p in unique_patients
}

bootstrap_results = []

for event_idx, event in enumerate(["Crackle", "Wheeze"]):

    y = y_val_multi[:, event_idx].astype(int)

    original_prob = original_val_prob[:, event_idx]
    aligned_prob = aligned_val_prob[:, event_idx]

    auc_differences = []
    ap_differences = []

    for b in range(n_bootstrap):

        sampled_patients = rng.choice(
            unique_patients,
            size=len(unique_patients),
            replace=True
        )

        indices = np.concatenate([
            patient_indices[p] for p in sampled_patients
        ])

        y_boot = y[indices]

        # Both classes must be present for ROC-AUC
        if len(np.unique(y_boot)) < 2:
            continue

        original_auc = roc_auc_score(
            y_boot, original_prob[indices]
        )

        aligned_auc = roc_auc_score(
            y_boot, aligned_prob[indices]
        )

        original_ap = average_precision_score(
            y_boot, original_prob[indices]
        )

        aligned_ap = average_precision_score(
            y_boot, aligned_prob[indices]
        )

        auc_differences.append(aligned_auc - original_auc)
        ap_differences.append(aligned_ap - original_ap)

    for metric, differences in [
        ("ROC-AUC", auc_differences),
        ("Average Precision", ap_differences)
    ]:

        differences = np.asarray(differences)

        bootstrap_results.append({
            "Event": event,
            "Metric": metric,
            "Valid bootstrap samples": len(differences),
            "Mean improvement": differences.mean(),
            "95% CI lower": np.percentile(differences, 2.5),
            "95% CI upper": np.percentile(differences, 97.5),
            "Bootstrap fraction positive": np.mean(differences > 0)
        })

bootstrap_df = pd.DataFrame(bootstrap_results)

display(bootstrap_df.round(4))


## 18. Aligned CNN–LSTM retraining


After applying the alignment transformation, the CNN–LSTM is retrained from fresh weights so that alignment effects are not confused with fine-tuning of the baseline model.


### 18.1 Fit aligned-training normalization


In [ ]:
# Calculate aligned derivative normalization statistics
aligned_delta_means = {}
aligned_delta_stds = {}

for channel in [1, 2]:

    channel_means = np.array([
        X_train_3ch_aligned_raw[:, f, :, channel][train_mfcc_mask].mean()
        for f in range(X_train_3ch_aligned_raw.shape[1])
    ])

    channel_stds = np.array([
        X_train_3ch_aligned_raw[:, f, :, channel][train_mfcc_mask].std()
        for f in range(X_train_3ch_aligned_raw.shape[1])
    ])

    channel_stds = np.maximum(channel_stds, 1e-6)

    aligned_delta_means[channel] = channel_means
    aligned_delta_stds[channel] = channel_stds

    print(
        f"Channel {channel}: "
        f"mean range=({channel_means.min():.5f}, "
        f"{channel_means.max():.5f}), "
        f"std range=({channel_stds.min():.5f}, "
        f"{channel_stds.max():.5f})"
    )


### 18.2 Prepare aligned training and validation tensors


In [ ]:
# Create independent copies
X_train_aligned_final = X_train_3ch_aligned_raw.copy()
X_val_aligned_final = X_val_3ch_aligned_raw.copy()

# Normalize derivative channels using aligned training statistics
for channel in [1, 2]:

    mean = aligned_delta_means[channel]
    std = aligned_delta_stds[channel]

    # Training normalization
    X_train_aligned_final[..., channel] = (
        X_train_aligned_final[..., channel]
        - mean[None, :, None]
    ) / std[None, :, None]

    # Validation uses training statistics
    X_val_aligned_final[..., channel] = (
        X_val_aligned_final[..., channel]
        - mean[None, :, None]
    ) / std[None, :, None]

    # Restore padding
    X_train_aligned_final[..., channel] *= (
        train_mfcc_mask[:, None, :]
    )

    X_val_aligned_final[..., channel] *= (
        val_mfcc_mask[:, None, :]
    )

print("Train shape:", X_train_aligned_final.shape)
print("Validation shape:", X_val_aligned_final.shape)

print("Train finite:", np.isfinite(X_train_aligned_final).all())
print("Validation finite:", np.isfinite(X_val_aligned_final).all())

for name, X, mask in [
    ("Aligned Train", X_train_aligned_final, train_mfcc_mask),
    ("Aligned Validation", X_val_aligned_final, val_mfcc_mask)
]:
    print(f"\n{name}")

    for channel, channel_name in [
        (0, "Log-Mel"),
        (1, "Delta"),
        (2, "Delta-Delta")
    ]:
        values = X[..., channel]
        valid_values = np.concatenate([
            values[i, :, mask[i]]
            for i in range(len(values))
        ])

        print(
            f"{channel_name}: "
            f"mean={valid_values.mean():.5f}, "
            f"std={valid_values.std():.5f}"
        )


### 18.3 Retrain the aligned CNN–LSTM from fresh weights


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, Model

# Build a fresh model with the ORIGINAL architecture
aligned_input = layers.Input(
    shape=(64, 157, 3),
    name="lung_sound_input"
)

x = layers.Conv2D(
    16, (3, 3), padding="same", activation="relu"
)(aligned_input)

x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(
    32, (3, 3), padding="same", activation="relu"
)(x)

x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(
    64, (3, 3), padding="same", activation="relu"
)(x)

x = layers.AveragePooling2D(
    pool_size=(4, 1)
)(x)

x = layers.Permute((2, 1, 3))(x)
x = layers.Reshape((39, 256))(x)

x = layers.LSTM(
    64,
    return_sequences=False,
    name="temporal_lstm"
)(x)

x = layers.Dropout(0.3)(x)

x = layers.Dense(
    32,
    activation="relu",
    name="cnn_lstm_dense"
)(x)

x = layers.Dropout(0.2)(x)

aligned_output = layers.Dense(
    2,
    activation="sigmoid",
    name="event_predictions"
)(x)

aligned_cnn_lstm = Model(
    inputs=aligned_input,
    outputs=aligned_output,
    name="device_aligned_cnn_lstm"
)

# Compile with the same loss and optimizer
aligned_cnn_lstm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

aligned_cnn_lstm.summary()


### 18.4 Save the aligned checkpoint


In [ ]:
aligned_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_cnn_lstm_device_aligned.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]

aligned_cnn_lstm_history = aligned_cnn_lstm.fit(
    X_train_aligned_final,
    y_train_multi,
    sample_weight=patient_weights_aligned,
    validation_data=(
        X_val_aligned_final,
        y_val_multi
    ),
    epochs=50,
    batch_size=32,
    callbacks=aligned_callbacks,
    verbose=1
)


## 19. Alignment results and robustness diagnostics


In [ ]:
import tensorflow as tf
import numpy as np
import soundfile as sf
import librosa
import matplotlib.pyplot as plt

# Load your best trained CNN-LSTM
model = tf.keras.models.load_model(
    "../models/best_cnn_lstm_patient_weighted.keras"
)

print("Model loaded successfully")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)


In [ ]:
import tensorflow as tf
import numpy as np
import soundfile as sf
import librosa
import matplotlib.pyplot as plt

model = tf.keras.models.load_model(
    "best_cnn_lstm_patient_weighted.keras",
    compile=False
)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)


In [ ]:
def reconstruct_segment_features(wav_path, start, end):

    # Same loading procedure as training
    audio, sr = librosa.load(
        wav_path,
        sr=TARGET_SR,
        mono=True
    )

    # Exact annotated segment
    start_sample = int(np.floor(start * sr))
    end_sample = int(np.ceil(end * sr))

    segment = audio[start_sample:end_sample]

    if len(segment) == 0:
        raise ValueError("Empty audio segment")

    # Match training: crop only if longer than target length
    if len(segment) > TARGET_SAMPLES:
        segment = fit_audio_length(segment, TARGET_SAMPLES)

    # Log-Mel extraction
    log_mel = extract_logmel(segment)

    actual_frames = log_mel.shape[1]

    # Reconstruct the original frame mask
    frame_mask = np.zeros(157, dtype=bool)

    if actual_frames < 157:
        pad_left = (157 - actual_frames) // 2
        frame_mask[pad_left:pad_left + actual_frames] = True
    else:
        frame_mask[:] = True

    log_mel = fit_logmel_frames(
        log_mel,
        target_frames=157
    )

    # Three-channel feature tensor
    X = np.zeros((64, 157, 3), dtype=np.float32)
    X[:, :, 0] = log_mel

    valid_mel = log_mel[:, frame_mask]
    n_valid = valid_mel.shape[1]

    width = min(
        9,
        n_valid if n_valid % 2 == 1 else n_valid - 1
    )

    if width < 3:
        raise ValueError("Insufficient frames for delta calculation")

    delta = librosa.feature.delta(
        valid_mel, width=width, order=1, axis=-1
    )

    delta_delta = librosa.feature.delta(
        valid_mel, width=width, order=2, axis=-1
    )

    X[:, :, 1][:, frame_mask] = delta
    X[:, :, 2][:, frame_mask] = delta_delta

    # Apply the training-set normalization
    for channel in [1, 2]:

        X[:, :, channel] -= delta_means[channel][:, None]
        X[:, :, channel] /= delta_stds[channel][:, None]
        X[:, :, channel] *= frame_mask[None, :]

    return X


### 19.1 Summarize the historical alignment comparison


In [ ]:
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

device_df = val_feature_order_df.copy()

# Extract device name from the recording filename
device_df["device"] = device_df["wav_path"].apply(
    lambda p: Path(p).stem.split("_")[-1]
)

device_df["crackle_true"] = y_val_multi[:, 0]
device_df["wheeze_true"] = y_val_multi[:, 1]

device_df["crackle_score"] = val_predictions[:, 0]
device_df["wheeze_score"] = val_predictions[:, 1]

records = []

for device, group in device_df.groupby("device"):

    for event in ["crackle", "wheeze"]:

        y_true = group[f"{event}_true"].values
        y_score = group[f"{event}_score"].values

        positives = int(np.sum(y_true == 1))
        negatives = int(np.sum(y_true == 0))

        if positives == 0 or negatives == 0:
            continue

        records.append({
            "device": device,
            "event": event,
            "samples": len(group),
            "positive_samples": positives,
            "negative_samples": negatives,
            "AUC": roc_auc_score(y_true, y_score),
            "AP": average_precision_score(y_true, y_score)
        })

device_comparison = pd.DataFrame(records)

display(
    device_comparison.sort_values(
        ["event", "device"]
    ).round(4)
)


In [ ]:
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

device_df = val_feature_order_df.copy()

# Extract device name from the recording filename
device_df["device"] = device_df["wav_path"].apply(
    lambda p: Path(p).stem.split("_")[-1]
)

device_df["crackle_true"] = y_val_multi[:, 0]
device_df["wheeze_true"] = y_val_multi[:, 1]

device_df["crackle_score"] = val_predictions[:, 0]
device_df["wheeze_score"] = val_predictions[:, 1]

records = []

for device, group in device_df.groupby("device"):

    for event in ["crackle", "wheeze"]:

        y_true = group[f"{event}_true"].values
        y_score = group[f"{event}_score"].values

        positives = int(np.sum(y_true == 1))
        negatives = int(np.sum(y_true == 0))

        if positives == 0 or negatives == 0:
            continue

        records.append({
            "device": device,
            "event": event,
            "samples": len(group),
            "positive_samples": positives,
            "negative_samples": negatives,
            "AUC": roc_auc_score(y_true, y_score),
            "AP": average_precision_score(y_true, y_score)
        })

device_comparison = pd.DataFrame(records)

display(
    device_comparison.sort_values(
        ["event", "device"]
    ).round(4)
)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

device_df = val_feature_order_df.copy()

device_df["device"] = device_df["wav_path"].apply(
    lambda p: Path(p).stem.split("_")[-1]
)

device_df["crackle_true"] = y_val_multi[:, 0]
device_df["wheeze_true"] = y_val_multi[:, 1]
device_df["crackle_score"] = val_predictions[:, 0]

# Keep only normal and crackle-only recordings
binary_df = device_df[
    binary_mask if "binary_mask" in globals()
    else (device_df["wheeze_true"] == 0)
].copy()

records = []

for device, group in binary_df.groupby("device"):

    labels = group["crackle_true"].values
    scores = group["crackle_score"].values

    positives = np.sum(labels == 1)
    negatives = np.sum(labels == 0)

    if positives == 0 or negatives == 0:
        continue

    records.append({
        "device": device,
        "samples": len(group),
        "normal_samples": negatives,
        "crackle_only_samples": positives,
        "crackle_AUC": roc_auc_score(labels, scores),
        "crackle_AP": average_precision_score(labels, scores),
        "AP_baseline": np.mean(labels)
    })

device_binary_comparison = pd.DataFrame(records)

display(device_binary_comparison.round(4))


In [ ]:
X_train_3ch = build_three_channel_features(
    X_train_saved,
    train_mask,
    "Training"
)

X_val_3ch = build_three_channel_features(
    X_val_saved,
    val_mask,
    "Validation"
)

print("\nThree-channel feature extraction complete.")


In [ ]:
def build_three_channel_features(X_logmel, frame_masks, split_name):
    n_samples, n_mels, n_frames, _ = X_logmel.shape

    X_3ch = np.zeros(
        (n_samples, n_mels, n_frames, 3),
        dtype=np.float32
    )

    X_3ch[:, :, :, 0] = X_logmel[:, :, :, 0]

    for i in range(n_samples):

        valid = frame_masks[i]
        valid_mel = X_logmel[i, :, :, 0][:, valid]

        n_valid = valid_mel.shape[1]

        if n_valid < 3:
            raise ValueError(f"Sample {i} has only {n_valid} valid frames")

        width = min(9, n_valid if n_valid % 2 == 1 else n_valid - 1)

        delta = librosa.feature.delta(
            valid_mel, width=width, order=1, axis=-1
        )

        delta_delta = librosa.feature.delta(
            valid_mel, width=width, order=2, axis=-1
        )

        # Index the 2D channel first to preserve (mel, time) orientation
        X_3ch[i, :, :, 1][:, valid] = delta
        X_3ch[i, :, :, 2][:, valid] = delta_delta

    print(f"\n{split_name} features built")
    print("Shape:", X_3ch.shape)
    print("Memory (MB):", round(X_3ch.nbytes / (1024**2), 2))

    return X_3ch


X_train_3ch = build_three_channel_features(
    X_train_saved, train_mask, "Training"
)

X_val_3ch = build_three_channel_features(
    X_val_saved, val_mask, "Validation"
)

print("\nThree-channel feature extraction complete.")


In [ ]:
# Store normalization statistics for reproducibility
delta_means = np.zeros((64, 2), dtype=np.float32)
delta_stds = np.zeros((64, 2), dtype=np.float32)

for channel in [1, 2]:

    channel_idx = channel - 1

    for f in range(64):

        valid_values = X_train_3ch[:, f, :, channel][train_mask]

        delta_means[f, channel_idx] = valid_values.mean()
        delta_stds[f, channel_idx] = max(
            valid_values.std(), 1e-6
        )

# Apply training statistics to both splits
for channel in [1, 2]:

    channel_idx = channel - 1

    for f in range(64):

        mean = delta_means[f, channel_idx]
        std = delta_stds[f, channel_idx]

        X_train_3ch[:, f, :, channel] = (
            X_train_3ch[:, f, :, channel] - mean
        ) / std

        X_val_3ch[:, f, :, channel] = (
            X_val_3ch[:, f, :, channel] - mean
        ) / std

# Restore padded positions to zero after normalization
X_train_3ch *= train_mask[:, np.newaxis, :, np.newaxis]
X_val_3ch *= val_mask[:, np.newaxis, :, np.newaxis]

print("NORMALIZATION COMPLETE")

for channel, name in [
    (0, "Log-Mel"),
    (1, "Delta"),
    (2, "Delta-Delta")
]:
    train_values = X_train_3ch[:, :, :, channel][
        np.broadcast_to(train_mask[:, None, :], (3529, 64, 157))
    ]

    val_values = X_val_3ch[:, :, :, channel][
        np.broadcast_to(val_mask[:, None, :], (496, 64, 157))
    ]

    print(f"\n{name}")
    print("Train mean:", round(float(train_values.mean()), 4))
    print("Train std:", round(float(train_values.std()), 4))
    print("Validation mean:", round(float(val_values.mean()), 4))
    print("Validation std:", round(float(val_values.std()), 4))

print("\nMaximum padded training value:",
      np.abs(X_train_3ch[~train_mask[:, None, :].repeat(64, axis=1)]).max())

print("Maximum padded validation value:",
      np.abs(X_val_3ch[~val_mask[:, None, :].repeat(64, axis=1)]).max())


In [ ]:
# Calculate class weights for the CURRENT training split

positive_counts_current = y_train_multi.sum(axis=0)
negative_counts_current = len(y_train_multi) - positive_counts_current

positive_weights_current = (
    negative_counts_current / positive_counts_current
)

print("CURRENT TRAINING SPLIT")

for i, event in enumerate(["Crackle", "Wheeze"]):
    print(f"\n{event}")
    print("Positive samples:", int(positive_counts_current[i]))
    print("Negative samples:", int(negative_counts_current[i]))
    print("Positive class weight:", round(float(positive_weights_current[i]), 4))

print("\nExisting positive_weights variable:")
if "positive_weights" in globals():
    print(positive_weights)
else:
    print("Not available")


In [ ]:
keras.utils.set_random_seed(42)

weighted_3ch_model = keras.Sequential([
    layers.Input(shape=X_train_3ch.shape[1:]),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.AveragePooling2D((4, 4)),

    layers.Flatten(),

    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(2, activation="sigmoid")
])

weighted_3ch_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(name="binary_accuracy")
    ]
)

weighted_3ch_model.summary()


In [ ]:
from tensorflow.keras import layers, Model

keras.utils.set_random_seed(42)

cnn_lstm_input = layers.Input(
    shape=(64, 157, 3),
    name="lung_sound_input"
)

x = layers.Conv2D(
    16, (3, 3), padding="same", activation="relu"
)(cnn_lstm_input)

x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(
    32, (3, 3), padding="same", activation="relu"
)(x)

x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(
    64, (3, 3), padding="same", activation="relu"
)(x)

x = layers.AveragePooling2D(
    pool_size=(4, 1)
)(x)

x = layers.Permute((2, 1, 3))(x)

x = layers.Reshape((39, 256))(x)

x = layers.LSTM(
    64,
    return_sequences=False,
    name="temporal_lstm"
)(x)

x = layers.Dropout(0.3)(x)

x = layers.Dense(
    32,
    activation="relu",
    name="cnn_lstm_dense"
)(x)

x = layers.Dropout(0.2)(x)

cnn_lstm_output = layers.Dense(
    2,
    activation="sigmoid",
    name="event_predictions"
)(x)

cnn_lstm_model = Model(
    inputs=cnn_lstm_input,
    outputs=cnn_lstm_output,
    name="lightweight_cnn_lstm"
)

cnn_lstm_model.summary()


In [ ]:
cnn_lstm_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(
            name="binary_accuracy"
        )
    ]
)

print("CNN-LSTM compiled successfully.")
print("Loss: Weighted Binary Cross-Entropy")
print("Optimizer: Adam")
print("Learning rate: 0.001")


In [ ]:
callbacks_cnn_lstm = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    )
]

history_cnn_lstm = cnn_lstm_model.fit(
    X_train_3ch,
    y_train_multi,
    sample_weight=patient_weights,
    validation_data=(
        X_val_3ch,
        y_val_multi
    ),
    epochs=50,
    batch_size=32,
    callbacks=callbacks_cnn_lstm,
    shuffle=True,
    verbose=1
)


In [ ]:
# ============================================================
# BUILD ALIGNED OFFICIAL TEST FEATURES
# ============================================================
# Copy original test Log-Mel features
X_test_logmel_aligned = X_test_3ch[..., 0:1].copy()
# Apply training-fitted device correction profiles
test_alignment_results = apply_device_alignment(
    X_test_logmel_aligned,
    test_feature_metadata,
    test_frame_mask,
    "Official Test"
)

print("\nTEST ALIGNMENT SUMMARY")
display(test_alignment_results.round(3))
# Reconstruct three-channel aligned test features
#    using the existing feature construction function
X_test_3ch_aligned_raw = build_three_channel_features(
    X_test_logmel_aligned,
    test_frame_mask,
    "Official Test Aligned"
)
# Apply aligned TRAINING-ONLY derivative normalization
X_test_aligned_final = X_test_3ch_aligned_raw.copy()

for channel in [1, 2]:

    X_test_aligned_final[..., channel] -= (
        aligned_delta_means[channel][None, :, None]
    )

    X_test_aligned_final[..., channel] /= (
        aligned_delta_stds[channel][None, :, None]
    )

    # Restore padded frames to zero
    X_test_aligned_final[..., channel] *= (
        test_frame_mask[:, None, :]
    )
# Integrity checks
assert X_test_aligned_final.shape == (2756, 64, 157, 3)
assert np.isfinite(X_test_aligned_final).all()

for channel in [1, 2]:
    padded_values = np.where(
        test_frame_mask[:, None, :],
        0.0,
        X_test_aligned_final[..., channel]
    )
    assert np.max(np.abs(padded_values)) < 1e-6

print("\nALIGNED OFFICIAL TEST FEATURES READY")
print("Shape:", X_test_aligned_final.shape)
print("All finite:", np.isfinite(X_test_aligned_final).all())
print("Range:", X_test_aligned_final.min(),
      X_test_aligned_final.max())
print("Original test features preserved:",
      X_test_3ch.shape == (2756, 64, 157, 3))


## 20. Final model checkpoint and evaluation outputs


### 20.1 Rebuild the documented inference input


In [ ]:
from pathlib import Path
import numpy as np
from tensorflow import keras
# Load the saved baseline checkpoint
baseline_path = Path("best_cnn_lstm_patient_weighted.keras")
assert baseline_path.is_file(), f"Checkpoint not found: {baseline_path.resolve()}"

demo_model = keras.models.load_model(
    baseline_path,
    compile=False
)
# Rebuild the original 3-channel test features
X_demo = build_three_channel_features(
    X_test_saved,
    test_mask,
    "Laptop inference demo"
)
# Apply the original training-fitted derivative statistics
for channel in [1, 2]:
    X_demo[..., channel] = (
        X_demo[..., channel]
        - delta_means[:, channel - 1][None, :, None]
    ) / delta_stds[:, channel - 1][None, :, None]

    X_demo[..., channel] *= test_mask[:, None, :]

assert X_demo.shape == (2756, 64, 157, 3)
assert np.isfinite(X_demo).all()
# Select the first example of each actual class
class_names = {
    0: "Normal",
    1: "Crackle",
    2: "Wheeze",
    3: "Crackle + Wheeze"
}

selected_indices = []
for class_id in range(4):
    matches = np.where(y_test_saved == class_id)[0]
    assert len(matches) > 0
    selected_indices.append(int(matches[0]))
# Run local inference
demo_inputs = X_demo[selected_indices]

demo_scores = demo_model.predict(
    demo_inputs,
    batch_size=4,
    verbose=0
)
# Convert scores into event labels using fixed thresholds
for index, scores in zip(selected_indices, demo_scores):

    crackle_score = float(scores[0])
    wheeze_score = float(scores[1])

    crackle = crackle_score >= 0.45
    wheeze = wheeze_score >= 0.60

    if crackle and wheeze:
        prediction = "Crackle + Wheeze"
    elif crackle:
        prediction = "Crackle"
    elif wheeze:
        prediction = "Wheeze"
    else:
        prediction = "Normal"

    print("\n------------------------------")
    print("Recording:", metadata_test.iloc[index]["recording"])
    print("Actual class:", class_names[int(y_test_saved[index])])
    print(f"Crackle score: {crackle_score:.4f}")
    print(f"Wheeze score:  {wheeze_score:.4f}")
    print("Predicted event:", prediction)


### 20.2 Evaluate the saved model on test predictions


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

all_test_scores = demo_model.predict(
    X_demo,
    batch_size=32,
    verbose=1
)

# Fixed thresholds used in the original evaluation
crackle_threshold = 0.45
wheeze_threshold = 0.60

y_pred_multi = np.column_stack([
    all_test_scores[:, 0] >= crackle_threshold,
    all_test_scores[:, 1] >= wheeze_threshold
]).astype(int)

# Event-wise metrics
for j, event in enumerate(["Crackle", "Wheeze"]):
    print(f"\n{event.upper()}")

    print("ROC-AUC:",
          round(roc_auc_score(y_test_multi[:, j],
                              all_test_scores[:, j]), 4))

    print("Average Precision:",
          round(average_precision_score(y_test_multi[:, j],
                                        all_test_scores[:, j]), 4))

    print("Precision:",
          round(precision_score(y_test_multi[:, j],
                                y_pred_multi[:, j]), 4))

    print("Recall:",
          round(recall_score(y_test_multi[:, j],
                             y_pred_multi[:, j]), 4))

    print("F1:",
          round(f1_score(y_test_multi[:, j],
                         y_pred_multi[:, j]), 4))

# Convert the two event outputs into four classes
y_pred_four = (
    y_pred_multi[:, 0] + 2 * y_pred_multi[:, 1]
)

# Class order: normal, crackle, wheeze, both
print("\nFOUR-CLASS CONFUSION MATRIX")
print(confusion_matrix(
    y_test_saved,
    y_pred_four,
    labels=[0, 1, 2, 3]
))

print("\nFOUR-CLASS REPORT")
print(classification_report(
    y_test_saved,
    y_pred_four,
    labels=[0, 1, 2, 3],
    target_names=[
        "Normal", "Crackle", "Wheeze", "Crackle + Wheeze"
    ],
    zero_division=0
))

print("Four-class accuracy:",
      round(np.mean(y_test_saved == y_pred_four), 4))

print("Four-class macro-F1:",
      round(f1_score(y_test_saved, y_pred_four,
                     average="macro"), 4))


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score
)

# IMPORTANT: Run in the original 649-cycle experiment's kernel.
assert X_val_3ch.shape == (649, 64, 157, 3), (
    f"Wrong validation experiment: {X_val_3ch.shape}. "
    "Do not use the 496-cycle validation set here."
)
assert y_val_multi.shape == (649, 2)

# Load the original saved baseline model
model_path = Path("best_cnn_lstm_patient_weighted.keras")
assert model_path.is_file(), f"Checkpoint not found: {model_path.resolve()}"

m0_threshold_model = keras.models.load_model(
    model_path, compile=False
)

# Generate validation scores
val_scores = m0_threshold_model.predict(
    X_val_3ch, batch_size=32, verbose=1
)

assert val_scores.shape == (649, 2)
assert np.isfinite(val_scores).all()

# Actual event labels
y_true = np.asarray(y_val_multi).astype(int)

# Four-class labels: Normal, Crackle, Wheeze, Both
y_true_four = y_true[:, 0] + 2 * y_true[:, 1]

# Calculate metrics for a threshold pair
def evaluate_threshold_pair(tc, tw):
    pred = np.column_stack([
        val_scores[:, 0] >= tc,
        val_scores[:, 1] >= tw
    ]).astype(int)

    pred_four = pred[:, 0] + 2 * pred[:, 1]

    return {
        "crackle_threshold": tc,
        "wheeze_threshold": tw,
        "crackle_F1": f1_score(y_true[:, 0], pred[:, 0], zero_division=0),
        "wheeze_F1": f1_score(y_true[:, 1], pred[:, 1], zero_division=0),
        "macro_F1": f1_score(
            y_true_four, pred_four,
            labels=[0, 1, 2, 3],
            average="macro", zero_division=0
        )
    }

# A. Existing fixed thresholds
fixed_result = evaluate_threshold_pair(0.45, 0.60)

# B. Independently optimise each event's F1
threshold_grid = np.round(np.arange(0.10, 0.901, 0.01), 2)

best_crackle = max(
    threshold_grid,
    key=lambda t: f1_score(
        y_true[:, 0], val_scores[:, 0] >= t,
        zero_division=0
    )
)

best_wheeze = max(
    threshold_grid,
    key=lambda t: f1_score(
        y_true[:, 1], val_scores[:, 1] >= t,
        zero_division=0
    )
)

independent_result = evaluate_threshold_pair(
    best_crackle, best_wheeze
)

# C. Jointly optimise four-class macro-F1
joint_results = [
    evaluate_threshold_pair(tc, tw)
    for tc in threshold_grid
    for tw in threshold_grid
]

joint_result = max(
    joint_results, key=lambda x: x["macro_F1"]
)

# Compare all three operating points
threshold_comparison = pd.DataFrame([
    {"strategy": "Existing fixed", **fixed_result},
    {"strategy": "Independent event F1", **independent_result},
    {"strategy": "Joint four-class macro-F1", **joint_result}
])

print("VALIDATION THRESHOLD COMPARISON")
display(threshold_comparison.round(4))

# Ranking metrics do not depend on thresholds
print("\nTHRESHOLD-INDEPENDENT METRICS")
for i, event in enumerate(["Crackle", "Wheeze"]):
    print(
        event,
        "ROC-AUC:", round(roc_auc_score(y_true[:, i], val_scores[:, i]), 4),
        "AP:", round(average_precision_score(y_true[:, i], val_scores[:, i]), 4)
    )


## Results snapshot: recorded development evidence

The table below preserves the historical validation aggregates recorded during model development. It is included to show the progression across model families; the values are not presented as a fresh rerun from the restored repository.

| Experiment | Validation accuracy | Validation macro F1 |
|---|---:|---:|
| Always-normal baseline | 0.5532 | 0.1781 |
| Compact log-Mel CNN | 0.4761 | 0.2008 |
| Time-resized log-Mel CNN | 0.4284 | 0.2352 |
| Frame-padded CNN | 0.4268 | 0.2424 |
| Weighted three-channel CNN | 0.4992 | 0.4079 |
| Patient-balanced three-channel CNN | 0.5193 | 0.4186 |
| CNN embeddings + RBF-SVM | 0.4484 | 0.3826 |
| CNN embeddings + k-NN (k=11) | 0.5208 | 0.4374 |
| CNN embeddings + bagged k-NN | 0.5239 | 0.4376 |
| **CNN–LSTM selected model** | **0.5655** | **0.5041** |

The CNN–LSTM gave the strongest recorded validation macro F1 among these development experiments. The evaluated test result was lower, which motivated the failure analysis rather than being treated as an isolated accuracy figure.

## ML contribution visible in this repository

The notebook documents the complete ML development path: dataset auditing, patient-separated splitting, feature engineering, several CNN representations, class weighting, patient balancing, classical classifiers on learned embeddings, CNN–LSTM temporal modelling, event-threshold selection, device-level diagnostics, and device-alignment experiments. The repository also retains the original full experiment notebook in `archive/full_notebooks/`, while this notebook is the cleaned primary narrative for reviewers and recruiters.

## 21. Final ML takeaway

The development process moved from simple spectral baselines to richer three-channel representations, patient-aware training, learned embeddings, temporal modelling, and device-variation analysis. The recorded CNN–LSTM produced the strongest validation macro F1 among the listed experiments, while the lower evaluated test performance exposed a generalization gap. The final analysis therefore focuses on what the model learned, where it failed, and which data and device factors may be contributing to those failures.

This is the central ML contribution of the project: not only training a classifier, but building and evaluating the full experimental pipeline needed to understand its behaviour.